In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 2.6 MB/s eta 0:00:00


In [ ]:
# # =================== In-text dataset span miner — v3.21 ===================
# # What’s new vs v3.20:
# #   • DA header matching is tolerant to letter-spaced ALLCAPS (e.g., "DATA AVAIL ABILIT Y …")
# #   • DA block end is relaxed so it won’t stop before the DOI line (e.g., right before "ORCID")
# #   • Canonicalize DOI inside returned in_text_span (remove internal spaces; always "https://doi.org/<core>")
# #   • Try DA block on BOTH pdfminer & PyPDF2 page views before other strategies
# #   • Keep all prior logic and structure, adding only the minimal new helpers and calls

# import sys, subprocess, shlex
# def _pip_install(pkg):
#     try:
#         __import__(pkg.split("==")[0].replace("-", "_"))
#     except Exception:
#         subprocess.check_call(shlex.split(f"{sys.executable} -m pip install -q {pkg}"))
# _pip_install("pdfminer.six")

# import os, re, logging, shutil, subprocess, unicodedata
# from pathlib import Path
# from dataclasses import dataclass
# from typing import List, Dict, Optional, Tuple
# from concurrent.futures import ProcessPoolExecutor, as_completed

# import pandas as pd
# from tqdm.auto import tqdm
# logging.getLogger("PyPDF2").setLevel(logging.ERROR)
# from PyPDF2 import PdfReader

# # ---- paths (adjust as needed) ----
# PDF_FOLDER = Path("/content/drive/MyDrive/Make_data_count_challenge/Data/train/PDF")
# CSV_PATH   = "/content/drive/MyDrive/Make_data_count_challenge/Data/train_labels_cleaned.csv"

# # For quick testing, keep your debug AID here; otherwise leave [] to process all
# DEBUG_ARTICLE_IDS = ["10.1002_ece3.6303"]
# SAVE_AS          = "in_text_spans_v321.csv"
# MAX_WORKERS      = max(1, (os.cpu_count() or 2) - 1)
# RECURSIVE_PDF    = True
# CTX_SENT_WIN     = 1
# MIN_SPAN_CHARS   = 180
# MAX_SPAN_CHARS   = 1200

# # -------------------- helpers --------------------
# def canon(s: str) -> str:
#     """Unicode-safe canonicalization: lower, strip accents, keep alnum only."""
#     if s is None: return ""
#     s = unicodedata.normalize("NFKD", s)
#     return "".join(ch for ch in s.lower() if ch.isalnum())

# def collapse_hyphen_breaks(s: str) -> str:
#     """
#     Normalize broken hyphens/newlines and DOI/url skeletons; also strip optional 'doi:' prefix.
#     """
#     s = s.replace("\r", "")
#     s = re.sub(r"-\s*\n\s*", "", s)
#     s = re.sub(r"(\b10)\s*\.\s*([0-9]{4,9})\s*/\s*", r"\1.\2/", s, flags=re.I)
#     s = re.sub(r"(?i)\bdoi\s*:\s*", "", s)      # normalize away 'doi:'
#     s = re.sub(r"(?i)doi\s*\.\s*org", "doi.org", s)
#     s = re.sub(r"(?i)dx\s*\.\s*doi\s*\.\s*org", "dx.doi.org", s)
#     s = re.sub(r"(?i)https?\s*:\s*/\s*/", "https://", s)
#     return s

# def normalize_ws_for_output(s: str) -> str:
#     """
#     Clean whitespace in the final span: join wrapped lines, drop NBSP/thin spaces, collapse doubles.
#     """
#     s = re.sub(r"\s*\n\s*", " ", s)
#     s = re.sub(r"[\u00A0\u2000-\u200B]", " ", s)  # nbsp & thin spaces
#     s = re.sub(r"[ \t]{2,}", " ", s)
#     return s.strip()

# def _normalize_doi_in_span(span: str, regs: List[re.Pattern]) -> str:
#     """
#     Replace any matched DOI (possibly with internal whitespace) by canonical 'https://doi.org/<core>'.
#     """
#     for R in regs:
#         m = R.search(span)
#         if m:
#             core = re.sub(r"\s+", "", (m.group(1) if m.groups() else m.group(0))).lower()
#             if core.startswith("10."):
#                 return span.replace(m.group(0), f"https://doi.org/{core}")
#     return span

# def find_pdf_for_article(article_id: str, pdf_folder: Path) -> Optional[Path]:
#     direct = pdf_folder / f"{article_id}.pdf"
#     if direct.exists():
#         return direct
#     variants = {article_id, article_id.replace("/", "_"), article_id.replace("_", "/")}
#     it = pdf_folder.rglob("*.pdf") if RECURSIVE_PDF else pdf_folder.glob("*.pdf")
#     for f in it:
#         if any(v in f.stem for v in variants):
#             return f
#     return None

# from typing import Tuple, List
# import shutil, subprocess
# from PyPDF2 import PdfReader
# from pdfminer.high_level import extract_pages
# from pdfminer.layout import LTTextContainer, LAParams
# import re, statistics

# def _text_quality_metrics(text: str) -> dict:
#     """Language-agnostic quality metrics (no hardcoded words)."""
#     letters = sum(ch.isalpha() for ch in text)
#     digits  = sum(ch.isdigit() for ch in text)
#     spaces  = text.count(" ")
#     tokens  = text.split()
#     avg_tok = statistics.mean([len(w) for w in tokens]) if tokens else 0.0
#     long_tok = sum(1 for w in tokens if len(w) >= 20)
#     # Letter→digit adjacency (e.g., 'at10') often spikes when spaces are lost.
#     l2d = len(re.findall(r"[A-Za-z][0-9]", text))
#     return {
#         "letters": letters, "digits": digits, "spaces": spaces,
#         "space_ratio": spaces / max(1, letters + digits),
#         "avg_token_len": avg_tok,
#         "long_tokens": long_tok,
#         "l2d": l2d,
#         "nonempty": any(ln.strip() for ln in text.splitlines())
#     }

# def _score_pages(pages: List[str]) -> float:
#     """Higher = better natural spacing/word boundaries."""
#     if not pages:
#         return -1e9
#     scores = []
#     for t in pages:
#         m = _text_quality_metrics(t)
#         if not m["nonempty"]:
#             scores.append(-1e9); continue
#         # weights chosen to strongly penalize space loss / fused words
#         s = (
#             100.0 * m["space_ratio"]          # prefer more spaces relative to letters+digits
#             - 1.5  * m["avg_token_len"]       # penalize long average tokens (fused words)
#             - 10.0 * (m["long_tokens"] / max(1, len(t.split())))  # penalize very long tokens
#             - 0.2  * m["l2d"]                 # mild penalty for letter→digit joins (e.g., "at10")
#         )
#         scores.append(s)
#     return sum(scores) / len(scores)

# def _try_pdftotext(pdf_path: str) -> Tuple[str, List[str]]:
#     if not shutil.which("pdftotext"):
#         return "", []
#     r = subprocess.run(
#         ["pdftotext", "-layout", pdf_path, "-"],
#         stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False
#     )
#     if not r.stdout:
#         return "", []
#     txt = r.stdout
#     pages = txt.split("\f") if "\f" in txt else [txt]
#     return "\n".join(pages), pages

# def _try_pdfminer(pdf_path: str) -> Tuple[str, List[str]]:
#     # LAParams tuned to preserve natural word spacing without gluing words
#     lap = LAParams(char_margin=2.0, word_margin=0.15, line_margin=0.5,
#                    boxes_flow=0.5, all_texts=True, detect_vertical=False)
#     pages = []
#     try:
#         for layout in extract_pages(pdf_path, laparams=lap):
#             buf = []
#             for el in layout:
#                 if isinstance(el, LTTextContainer):
#                     buf.append(el.get_text())
#             pages.append("".join(buf))
#     except Exception:
#         return "", []
#     return "\n".join(pages), pages

# def _try_pypdf2(pdf_path: str) -> Tuple[str, List[str]]:
#     try:
#         reader = PdfReader(pdf_path, strict=False)
#         pages = []
#         for page in reader.pages:
#             try:
#                 t = page.extract_text()
#             except Exception:
#                 t = None
#             pages.append(t or "")
#         return ("\n".join(pages), pages) if any(p.strip() for p in pages) else ("", [])
#     except Exception:
#         return "", []

# def extract_text_with_pages(pdf_path: str) -> Tuple[str, List[str]]:
#     """
#     Run multiple extractors and pick the highest-quality text (no word-specific hacks).
#     Preference is data-driven via spacing/word-boundary metrics.
#     """
#     candidates = []

#     # 1) pdftotext -layout (often closest to "as-rendered" paragraphs)
#     full, pages = _try_pdftotext(pdf_path)
#     if pages:
#         candidates.append(("pdftotext", full, pages, _score_pages(pages)))

#     # 2) pdfminer.six with tuned LAParams
#     full, pages = _try_pdfminer(pdf_path)
#     if pages:
#         candidates.append(("pdfminer", full, pages, _score_pages(pages)))

#     # 3) PyPDF2 (fast, but often loses spaces on ACS/Elsevier)
#     full, pages = _try_pypdf2(pdf_path)
#     if pages:
#         candidates.append(("pypdf2", full, pages, _score_pages(pages)))

#     if not candidates:
#         return "", []

#     # Pick the extractor with the best score
#     best = max(candidates, key=lambda x: x[3])
#     return best[1], best[2]

# # ---- NEW: get pages from specific engines (for DA-first on both) ----
# def _pages_via_pypdf2(pdf_path: str) -> List[str]:
#     try:
#         reader = PdfReader(pdf_path, strict=False)
#         return [(p.extract_text() or "") for p in reader.pages]
#     except Exception:
#         return []

# def _pages_via_pdfminer(pdf_path: str) -> List[str]:
#     try:
#         lap = LAParams(char_margin=2.0, word_margin=0.15, line_margin=0.5,
#                        boxes_flow=0.5, all_texts=True, detect_vertical=False)
#         pages = []
#         for layout in extract_pages(pdf_path, laparams=lap):
#             buf = []
#             for el in layout:
#                 if isinstance(el, LTTextContainer):
#                     buf.append(el.get_text())
#             pages.append("".join(buf))
#         return pages
#     except Exception:
#         return []

# def extract_link_uris_by_page(pdf_path: str) -> Dict[int, List[str]]:
#     uris_by_page = {}
#     try:
#         reader = PdfReader(pdf_path, strict=False)
#         for i, page in enumerate(reader.pages):
#             uris = []
#             annots = page.get("/Annots")
#             if annots:
#                 for a in annots:
#                     try:
#                         obj = a.get_object()
#                         if "/A" in obj and "/URI" in obj["/A"]:
#                             uris.append(str(obj["/A"]["/URI"]))
#                         if "/URI" in obj:
#                             uris.append(str(obj["/URI"]))
#                     except Exception:
#                         continue
#             uris_by_page[i] = uris
#     except Exception:
#         pass
#     return uris_by_page

# def text_to_lines(text: str) -> List[str]:
#     return text.splitlines()

# def split_sentences(text: str) -> List[str]:
#     # light sentence splitter with common abbreviations protected
#     t = re.sub(r"[ \t]+", " ", text)
#     for a in ["et al.","e.g.","i.e.","Dr.","Prof.","Fig.","Eq.","Ref.","Refs.","No.","Vol.","pp.","Inc.","Ltd."]:
#         t = t.replace(a, a.replace(".", "◊"))
#     parts = re.split(r"(?<=[\.\?\!])\s+(?=[A-Z0-9\[])|(?<=\.)\n+", t)
#     return [p.replace("◊",".").strip() for p in parts if p and p.strip()]

# def paragraph_bounds(full_text: str, pos: int) -> Tuple[int, int]:
#     left = full_text.rfind("\n\n", 0, pos)
#     right = full_text.find("\n\n", pos)
#     if left == -1: left = 0
#     else: left = left + 2
#     if right == -1: right = len(full_text)
#     return (left, right)

# def find_sentence_span(text: str, match_start: int, ctx_sent_win: int = 1) -> Tuple[int,int,str,str,str]:
#     sents = split_sentences(text)
#     spans, off = [], 0
#     for sent in sents:
#         idx = text.find(sent, off)
#         if idx < 0:
#             idx = text[off:].find(sent)
#             if idx >= 0: idx += off
#         if idx >= 0:
#             spans.append((idx, idx+len(sent))); off = idx+len(sent)
#         else:
#             spans.append((off, off+len(sent))); off += len(sent)
#     si = 0
#     for i,(a,b) in enumerate(spans):
#         if a <= match_start < b:
#             si = i; break
#     li = max(0, si-ctx_sent_win); ri = min(len(sents)-1, si+ctx_sent_win)
#     left  = " ".join(sents[li:si]) if si>li else ""
#     exact = sents[si]
#     right = " ".join(sents[si+1:ri+1]) if ri>si else ""
#     return spans[si][0], spans[si][1], exact.strip(), left.strip(), right.strip()

# def expand_small_span(text: str, start: int, end: int, min_chars: int, max_chars: int) -> Tuple[int,int,str]:
#     if end - start >= min_chars:
#         return start, end, text[start:end].strip()
#     pL, pR = paragraph_bounds(text, start)
#     if (pR - pL) >= min_chars:
#         blob = text[pL:pR].strip()
#         return pL, pR, (blob[:max_chars].rsplit(" ",1)[0] + " …") if len(blob)>max_chars else blob
#     half = max(min_chars//2, 160)
#     L = max(0, start-half); R = min(len(text), end+half)
#     snip = text[L:R].strip()
#     return L, R, (snip[:max_chars].rsplit(" ",1)[0] + " …") if len(snip)>max_chars else snip

# def guess_section_from_lines(lines: List[str], full_text: str, match_start: int) -> str:
#     # Walk upward to nearest short ALLCAPS or known heading
#     cum, total = [], 0
#     for ln in lines: cum.append(total); total += len(ln)+1
#     import bisect
#     li = max(0, min(len(lines)-1, bisect.bisect_right(cum, match_start)-1))
#     for j in range(li, max(-1, li-30), -1):
#         L = lines[j].strip()
#         if not L: continue
#         if len(L) <= 80 and (
#             L.isupper()
#             or re.match(r"(?i)^(data|data availability|availability|data accessibility|associated|supporting|supplementary|references|methods|materials|acknowledg|appendix|materials and methods|results|discussion|conclusions)", L)
#         ):
#             return L
#     return ""

# # -------------------- dataset id parsing --------------------
# @dataclass
# class DatasetIdInfo:
#     raw: str
#     lower: str
#     base_doi: Optional[str]
#     version: Optional[str]
#     repo_hint: Optional[str]
#     canon_target: str
#     canon_suffix: str
#     zenodo_id: Optional[str]

# def _infer_repo(lower_id: str) -> Optional[str]:
#     if "dryad" in lower_id or "10.5061" in lower_id: return "Dryad"
#     if "tcia" in lower_id or "cancerimagingarchive" in lower_id or "10.7937" in lower_id: return "TCIA"
#     if "mendeley" in lower_id or "10.17632" in lower_id: return "Mendeley Data"
#     if "pasta" in lower_id or "10.6073" in lower_id: return "PASTA/LTER"
#     if "cranfield.rd" in lower_id or "10.17862" in lower_id: return "Cranfield RD"
#     if "10.11583" in lower_id or "dtu." in lower_id: return "DTU Data"
#     if "10.6075" in lower_id: return "UCSD Library Digital Collections"
#     if "figshare" in lower_id or "10.6084" in lower_id: return "figshare"
#     if "zenodo" in lower_id or "10.5281" in lower_id: return "Zenodo"
#     return None

# def parse_dataset_id(dataset_id: str) -> DatasetIdInfo:
#     s = str(dataset_id).strip(); lower = s.lower(); doi = None
#     if lower.startswith(("http://","https://")):
#         lower_no_proto = re.sub(r"^https?://", "", lower)
#         lower_no_proto = re.sub(r"^(dx\.)?doi\.org/", "", lower_no_proto)
#         m = re.search(r"(10\.\d{4,9}/[^\s]+)", lower_no_proto)
#         if m: doi = m.group(1)
#     elif lower.startswith("10."):
#         doi = lower
#     base_doi, version = None, None
#     if doi:
#         doi = doi.strip().strip(".,;")
#         m = re.match(r"^(10\.[^ ]+?)(\.v\d+)$", doi)
#         if m: base_doi, version = m.group(1), m.group(2)
#         else: base_doi = doi
#     repo_hint = _infer_repo(lower if not doi else doi)
#     suffix = base_doi.split("/",1)[-1] if base_doi else lower.split("/",1)[-1]
#     zenodo_id = None
#     if (base_doi or "").startswith("10.5281/zenodo."):
#         zenodo_id = (base_doi or lower).split("zenodo.",1)[-1].split("/",1)[0].split(".v",1)[0]
#     return DatasetIdInfo(
#         raw=s, lower=lower, base_doi=base_doi, version=version, repo_hint=repo_hint,
#         canon_target=canon(base_doi or lower), canon_suffix=canon(suffix or lower), zenodo_id=zenodo_id
#     )

# # ---- whitespace-tolerant DOI & repo URL detection ----
# def _loose_tail_regex(tail: str) -> str:
#     """
#     Build a regex that tolerates arbitrary internal whitespace around ./;:_- in the DOI tail.
#     """
#     pat = []
#     for ch in tail:
#         if ch.isalnum():
#             pat.append(f"{re.escape(ch)}\\s*")
#         elif ch in "./:_-;":
#             pat.append(f"\\s*{re.escape(ch)}\\s*")
#         else:
#             pat.append(f"\\s*{re.escape(ch)}\\s*")
#     return "".join(pat)

# def doi_regex_list_for(info: DatasetIdInfo) -> List[re.Pattern]:
#     regs = []
#     if info.base_doi:
#         tail = info.base_doi.split("/",1)[-1]
#         tail_loose = _loose_tail_regex(tail)
#         regs.append(re.compile(
#             r"(?:doi\s*:?\s*)?"                                 # accept optional 'doi:'
#             r"(?:https?://(?:dx\.)?doi\.org/\s*)?"
#             r"(10\s*\.\s*[0-9\s]{4,9}\s*/\s*" + tail_loose + r")"
#             r"(?:\s*\.v\d+)?",
#             flags=re.IGNORECASE | re.DOTALL
#         ))
#     # repo-specific alternates (e.g., Zenodo records)
#     if info.repo_hint == "Zenodo" and info.zenodo_id:
#         regs.append(re.compile(rf"zenodo\.org/\s*record[s]?\s*/\s*{re.escape(info.zenodo_id)}", re.I))
#         regs.append(re.compile(rf"zenodo[^0-9]{{0,12}}{re.escape(info.zenodo_id)}", re.I))
#     # Dryad sometimes appears as "datadryad.org/stash/dataset/doi:10.5061/dryad.x"
#     if info.repo_hint == "Dryad" and info.base_doi:
#         tail = info.base_doi.split("/",1)[-1]
#         regs.append(re.compile(rf"datadryad\.org/.+10\.?\s*5061\s*/\s*{_loose_tail_regex(tail.split('/',1)[-1])}", re.I))
#         regs.append(re.compile(rf"dryad[^/]*[:\s]+10\.?\s*5061\s*/\s*{_loose_tail_regex(tail)}", re.I))
#     return regs

# def normalize_captured_doi(blob: str) -> str:
#     """
#     From a DOI-like capture that may contain spaces, rebuild clean "10.xxxx/..." (lowercased, strip trailing punct).
#     """
#     if not blob: return ""
#     z = re.sub(r"\s+", "", blob).lower()
#     z = z.rstrip(").,;:[]")
#     m = re.search(r"10\.\d{4,9}/.+", z)
#     return m.group(0) if m else z

# # -------------------- footnote & superscript mapping --------------------
# FOOTNOTE_LINE_RE = re.compile(r"^\s*(\d{1,2})[\)\].]?\s*(.*)$")

# def split_main_vs_footnote_lines(page_text: str) -> Tuple[List[str], List[str]]:
#     lines = [ln for ln in page_text.splitlines() if ln.strip()]
#     if not lines: return [], []
#     # heuristic: numbered block near bottom
#     foot_start = None
#     for i in range(len(lines)-1, max(-1, len(lines)-15), -1):
#         if FOOTNOTE_LINE_RE.match(lines[i].strip()):
#             if i == 0 or not lines[i-1].strip() or FOOTNOTE_LINE_RE.match(lines[i-1].strip()):
#                 foot_start = i if not FOOTNOTE_LINE_RE.match(lines[i-1].strip()) else i-1
#                 break
#     if foot_start is None:
#         foot_start = max(0, len(lines)-6)
#     return lines[:foot_start], lines[foot_start:]

# def get_footnote_entry_for_dataset(page_text: str, info: DatasetIdInfo) -> Tuple[Optional[int], Optional[str]]:
#     _, foot = split_main_vs_footnote_lines(page_text)
#     tokens = candidate_canon_tokens(info)
#     for ln in foot:
#         m = FOOTNOTE_LINE_RE.match(ln.strip())
#         if not m: continue
#         num = int(m.group(1)); rest = m.group(2)
#         c = canon(rest)
#         if any(tok in c for tok in tokens):
#             return num, ln.strip()
#     return None, None

# def find_reference_sentence_for_marker(page_text: str, marker_num: int) -> Tuple[Optional[str], Optional[str]]:
#     main, _ = split_main_vs_footnote_lines(page_text)
#     if not main: main = [ln for ln in page_text.splitlines() if ln.strip()]
#     page_join = "\n".join(main)
#     n = str(marker_num)
#     pats = [rf"(?<=\w){n}(?!\d)", rf"\({n}\)", rf"\[{n}\]", rf"\s{n}\s"]
#     kw = ["dataset","data","release","available","download","repository","zenodo","dryad","figshare","tcia"]
#     best_line, best_score = "", 0
#     for ln in main:
#         low = ln.lower(); score = 0
#         if any(re.search(p, ln) for p in pats): score += 5
#         if any(k in low for k in kw): score += 2
#         if 40 <= len(ln) <= 280: score += 1
#         if score > best_score: best_line, best_score = ln, score
#     if not best_line: return None, None
#     pos = page_join.find(best_line[:80])
#     if pos < 0: pos = max(0, page_join.lower().find(best_line.strip().lower()[:40]))
#     s0, e0, exact, _, _ = find_sentence_span(page_join, max(0,pos), CTX_SENT_WIN)
#     _, _, span = expand_small_span(page_join, s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)
#     return span, best_line

# # -------------------- candidate tokens & finders --------------------
# def candidate_canon_tokens(info: DatasetIdInfo) -> List[str]:
#     """
#     Canonical tokens to look for in alnum-only text (canon(page)).
#     """
#     toks = []
#     if info.canon_target: toks.append(info.canon_target)
#     if info.canon_suffix: toks.append(info.canon_suffix)
#     if info.repo_hint == "Zenodo" and info.zenodo_id:
#         toks.append(canon("zenodo" + info.zenodo_id))
#         toks.append(canon("zenodoorgrecord" + info.zenodo_id))
#         toks.append(canon("zenodoorgrecords" + info.zenodo_id))
#     return list(dict.fromkeys([t for t in toks if t]))

# def sentence_containing_id(page_text: str, info: DatasetIdInfo) -> Optional[Tuple[int,int,str,str]]:
#     """
#     Return (start, end, sentence, matched_string) for the shortest sentence containing the dataset id
#     via loose DOI or repo-URL synonyms.
#     """
#     clean = collapse_hyphen_breaks(page_text)
#     regs = doi_regex_list_for(info)
#     hit = None
#     for R in regs:
#         m = R.search(clean)
#         if m:
#             matched = m.group(1) if m.groups() else m.group(0)
#             hit = (m.start(), m.end(), matched); break
#     if not hit:
#         return None
#     pos = hit[0]
#     s0, e0, sent, _, _ = find_sentence_span(clean, pos, CTX_SENT_WIN)
#     return (s0, e0, sent, hit[2])

# # --- NEW: letter-spaced ALLCAPS collapse for headings ---
# def unsplit_spaced_caps(line: str) -> str:
#     # Turn "D A T A  A V A I L ABILIT Y  S TATEMENT" into "DATA AVAILABILITY STATEMENT"
#     return re.sub(r'(?<![A-Za-z])(?:[A-Z]\s+){2,}[A-Z](?![A-Za-z])',
#                   lambda m: m.group(0).replace(' ', ''), line)

# # --- 5) Tolerant page-heading scan when expanding a sentence to a block
# def block_span_from_headings(page_text: str, sent_start: int, sent_end: int) -> Tuple[int,int]:
#     clean = collapse_hyphen_breaks(page_text)
#     start = max(0, sent_start); end = min(len(clean), sent_end)
#     look_back = clean[max(0, start-1600):start]
#     head_pats = [
#         r"(?i)(?:^|[\n\r])\s*(?:■\s*)?ASSOCIATED\W*CONTENT",
#         r"(?i)(?:^|[\n\r])\s*Data\W*Availability(?:\W*Statement)?",
#         r"(?i)(?:^|[\n\r])\s*Availability\W*of\W*Data(?:\W*and\W*Materials)?",
#         r"(?i)(?:^|[\n\r])\s*Data\W*Accessibility",
#         r"(?i)(?:^|[\n\r])\s*Data\W*and\W*materials\W*availability",
#         r"(?i)(?:^|[\n\r])\s*SUPPORTING\W*INFORMATION"
#     ]
#     candidates = []
#     for kw in head_pats:
#         m = list(re.finditer(kw, look_back, flags=re.IGNORECASE))
#         if m:
#             candidates.append(max(0, start-1600) + m[-1].start())
#     block_start = max(candidates) if candidates else start
#     return (block_start, end)

# # -------------------- STEP-2 type classifier --------------------
# def classify_source_type(anchor_sentence: str, section_heading: str, has_footnote: bool) -> Tuple[str, str]:
#     """
#     Returns (type_code, type_label) mapping to your 2-1…2-16 taxonomy.
#     """
#     s = anchor_sentence or ""
#     h = section_heading or ""
#     s_low = s.lower(); h_low = h.lower()
#     def has(p): return re.search(p, s, flags=re.I)

#     if has_footnote:
#         return ("2-3", "Footnote or endnote (superscript)")

#     if re.search(r"(?i)\bdata\s+availability\b|\bavailability\s+of\s+data\b|\bdata\s+accessibilit", h) \
#        or re.search(r"(?i)\bdata\s+availability\b", s):
#         return ("2-5", "Data availability section")

#     if re.match(r"(?i)ASSOCIATED\s+CONTENT", h) or "supporting information" in h_low:
#         return ("2-6", "Supplementary material reference")

#     if re.search(r"\[\d+\]|\(\d+\)", s):
#         return ("2-2", "Numeric citation (Vancouver/IEEE)")

#     if re.search(r"https?://|doi\.org", s_low):
#         return ("2-4", "Inline link (URL or DOI in text)")

#     if re.search(r"(?i)\bfigure\b|\bfig\.\b|\btable\b|^figure\s+\d+|^fig\.", s):
#         return ("2-9", "Table or figure caption")

#     if re.search(r"(?i)acknowledg", h) or re.search(r"(?i)acknowledg", s):
#         return ("2-7", "Acknowledgments")

#     if re.search(r"(?i)methods|materials|experimental|methodology", h) or re.search(r"(?i)\bwe used\b|\bwe trained\b", s_low):
#         return ("2-8", "Methods section (descriptive mention)")

#     if re.search(r"(?i)\baccession\b|\bprjna\b|\bpdb\b|\bgse\d+", s):
#         return ("2-10", "Repository or accession ID mention")

#     if re.search(r"(?i)dataset|data set|repository|zenodo|dryad|figshare|tcia|mendeley|ncbi", s_low):
#         return ("2-11", "In-text narrative with repository name")

#     # Author–date hint: "Smith (2022)" style
#     if re.search(r"\([1-2][0-9]{3}\)", s) and re.search(r"[A-Z][a-z].+\([1-2][0-9]{3}\)", s):
#         return ("2-1", "In-text citation (author–date style)")

#     return ("2-16", "OTHER WAY")

# # --- NEW: high-priority data-availability / associated-content locator ---
# DA_HEAD_PATTERNS = [
#     r"(?i)^\s*Data\W*Availability(?:\W*Statement)?\s*$",
#     r"(?i)^\s*Availability\W*of\W*Data(?:\W*and\W*Materials)?\s*$",
#     r"(?i)^\s*Data\W*Accessibility\s*$",
#     r"(?i)^\s*Data\W*and\W*materials\W*availability\s*$",
#     r"(?i)^\s*(?:■\s*)?ASSOCIATED\W*CONTENT\s*$",      # tolerate bullet + no/odd spaces
#     r"(?i)^\s*SUPPORTING\W*INFORMATION\s*$",
# ]

# def _page_offsets(pages: List[str]) -> List[Tuple[int,int]]:
#     """Return [(start,end)) offsets of each page within the joined full_text."""
#     offs = []; pos = 0
#     for p in pages:
#         offs.append((pos, pos + len(p)))
#         pos += len(p) + 1  # account for join newline
#     return offs

# def _find_section_blocks_on_page(page_text: str) -> List[Tuple[int,int,str]]:
#     """
#     Return list of (start_idx, end_idx, heading_text) for DA-like blocks on this page_text.
#     Relaxed: if the next ALLCAPS header is immediately followed by DOI/repo hints, do not stop yet.
#     """
#     page_clean = collapse_hyphen_breaks(page_text)
#     lines = page_clean.splitlines(True)  # keep newlines
#     idxs = []
#     for i, ln in enumerate(lines):
#         chk = unsplit_spaced_caps(ln)          # normalized line for matching
#         for pat in DA_HEAD_PATTERNS:
#             if re.match(pat, chk.strip()):
#                 j = i + 1
#                 while j < len(lines):
#                     L = unsplit_spaced_caps(lines[j]).strip()
#                     if not L and j+1 < len(lines) and not lines[j+1].strip():
#                         break
#                     # relaxed stopper: allow one header if the following lines contain DOI/repo tokens
#                     next_chunk = "".join(lines[j:j+3])
#                     if (len(L) <= 80 and (L.isupper() or re.match(
#                         r"(?i)^\s*(references|acknowledg|author information|notes|conclusions|appendix|methods|materials|results|discussion|orcid)\b", L))
#                         and not re.search(r"(?:doi\s*:?\s*)?(?:https?://)?doi\.org/10\.\d{4,9}/|10\.\d{4,9}/|dryad|figshare|zenodo", next_chunk, re.I)):
#                         break
#                     j += 1
#                 start = sum(len(x) for x in lines[:i])
#                 end   = sum(len(x) for x in lines[:j])
#                 idxs.append((start, end, chk.strip()))
#                 break
#     return idxs

# def _match_in_da_blocks(pages: List[str], info: DatasetIdInfo) -> Optional[Dict]:
#     """
#     Scan pages for DA/Associated Content blocks that contain the dataset DOI or repo-URL synonyms.
#     Return a populated result dict if found.
#     """
#     regs = doi_regex_list_for(info)
#     for pi, page in enumerate(pages):
#         page_clean = collapse_hyphen_breaks(page)
#         blocks = _find_section_blocks_on_page(page_clean)
#         if not blocks:
#             continue
#         for (a, b, heading_txt) in blocks:
#             block = page_clean[a:b].strip()
#             # Within "ASSOCIATED CONTENT", prefer the sub-block starting at "Data Availability"
#             if re.match(r"(?i)^\s*ASSOCIATED\s+CONTENT\s*$", heading_txt):
#                 sub = re.search(r"(?im)^\s*Data\s+Availability(?:\s+Statement)?\s*$", block)
#                 if sub:
#                     a = a + sub.start()
#                     block = page_clean[a:b].strip()
#                     heading_txt = "Data Availability Statement"

#             # Check presence of target id
#             tok_hit = any(tok in canon(block) for tok in candidate_canon_tokens(info))
#             reg_hit = any(R.search(block) for R in regs)
#             if tok_hit or reg_hit:
#                 type_code, type_label = classify_source_type(block.splitlines()[0], heading_txt, False)
#                 dataset_in_paper = ""
#                 for R in regs:
#                     m = R.search(block)
#                     if m:
#                         g = m.group(1) if m.groups() else m.group(0)
#                         d = normalize_captured_doi(g)
#                         if d.startswith("10."):
#                             dataset_in_paper = d
#                             break

#                 arrow = f"Footnote: (none)\n    → Anchor: {block}"
#                 return {
#                     "in_text_span": block,
#                     "anchor_sentence": block.splitlines()[0].strip(),
#                     "footnote_number": "",
#                     "footnote_text": "",
#                     "page_index": str(pi),
#                     "match_label": "da_hit_priority",
#                     "match_confidence": 0.98,
#                     "dataset_in_paper": dataset_in_paper or (info.base_doi or ""),
#                     "relation_hint": "availability/archival",
#                     "span_source": f"page_{pi}",
#                     "arrow_chain": arrow,
#                     "debug": "data_availability_priority",
#                     "section_guess": heading_txt or "Data Availability",
#                     "source_type": "2-5",
#                     "source_type_label": "Data availability section",
#                 }
#     return None

# # -------------------- matching strategy --------------------
# def page_has_candidate(info: DatasetIdInfo, page_text: str, page_links: List[str]) -> Tuple[bool, str]:
#     cpage = canon(page_text)
#     toks = candidate_canon_tokens(info)
#     in_text = any(tok in cpage for tok in toks)
#     in_links = any(any(tok in canon(u) for tok in toks) for u in page_links or [])
#     src = "text" if in_text else ("link" if in_links else "")
#     return (in_text or in_links, src)

# def _truncate_before_references(full_text: str) -> str:
#     m = re.search(r"(?im)^\s*(references|literature\s+cited|bibliography)\s*$", full_text)
#     return full_text[:m.start()] if m else full_text

# # --- NEW: pick best span among multiple DOI hits (longest descriptive paragraph) ---
# def _best_doi_span(page_text: str, regs: List[re.Pattern]) -> Optional[str]:
#     candidates = []
#     clean_page = collapse_hyphen_breaks(page_text)
#     for R in regs:
#         for m in R.finditer(clean_page):
#             s0, e0, exact, _, _ = find_sentence_span(clean_page, m.start(), CTX_SENT_WIN)
#             L, R_, span = expand_small_span(clean_page, s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)
#             score = len(span)
#             if re.search(r"(?i)dataset|segmentation|collection|metadata|radiomics|thoracic|effusion", span):
#                 score += 300
#             candidates.append((score, span, exact))
#     if not candidates:
#         return None
#     _, best_span, _ = max(candidates, key=lambda x: x[0])
#     return best_span.strip()

# # --- NEW: references parsing for DOI → numeric citation hop ---
# REF_SPLIT_RE = re.compile(r"(?m)^\s*(\d{1,3})\.\s+")
# def _extract_references_block(full_text: str) -> Optional[str]:
#     m = re.search(r"(?im)^\s*(references|bibliography)\s*$", full_text)
#     return full_text[m.start():] if m else None

# def _reference_index_for_doi(full_text: str, regs: List[re.Pattern]) -> Optional[int]:
#     refs = _extract_references_block(collapse_hyphen_breaks(full_text))
#     if not refs:
#         return None
#     parts = REF_SPLIT_RE.split(refs)
#     for i in range(1, len(parts), 2):
#         try: idx = int(parts[i])
#         except Exception: continue
#         blob = parts[i+1]
#         for R in regs:
#             if R.search(blob):
#                 return idx
#     return None

# def _best_body_span_for_reference(full_text: str, ref_idx: int) -> Optional[Tuple[str, str, str]]:
#     body = _truncate_before_references(full_text)
#     pat = re.compile(rf"(?<!\d)(?:\[{ref_idx}\]|\({ref_idx}\)|{ref_idx})(?!\d)")
#     hits = [m.start() for m in pat.finditer(body)]
#     if not hits:
#         return None
#     tokens = r"(dataset|data set|collection|tcia|zenodo|dryad|figshare|nsclc|radiomics)"
#     bad_sections = re.compile(r"(?i)acknowledg|funding|support|conflict|note")

#     best = None; best_score = -1
#     for pos in hits:
#         s0, e0, exact, _, _ = find_sentence_span(body, pos, CTX_SENT_WIN)
#         L, R_, span = expand_small_span(body, s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)
#         lines_all = text_to_lines(body)
#         section_guess = guess_section_from_lines(lines_all, body, s0)
#         if bad_sections.search(section_guess): continue
#         score = 0
#         if re.search(tokens, span, flags=re.I): score += 3
#         if re.search(tokens, exact, flags=re.I): score += 2
#         if 250 <= len(span) <= 1000: score += 1
#         if re.search(r"(?i)introduction|methods|materials|results|discussion|conclusion", section_guess): score += 2
#         if score > best_score:
#             best_score = score
#             best = (span.strip(), exact.strip(), section_guess)
#     return best


# def match_dataset(full_text: str, pages: List[str], uris_by_page: Dict[int, List[str]], info: DatasetIdInfo) -> Dict:
#     """
#     Build match record with sentence/block + classification.
#     """
#     result = {
#         "in_text_span":"NOT FOUND","anchor_sentence":"","footnote_number":"",
#         "footnote_text":"","page_index":"","section_guess":"","match_label":"",
#         "match_confidence":0.0,"repo_guess":info.repo_hint or "","dataset_in_paper":"",
#         "version_mismatch":"","relation_hint":"unknown","span_source":"", "arrow_chain":"",
#         "debug":"", "source_type":"", "source_type_label":""
#     }

#     regs = doi_regex_list_for(info)

#     # NEW: 0) High-priority: Data Availability / Associated Content blocks that contain the target id (best extractor pages only)
#     da_first = _match_in_da_blocks(pages, info)
#     if da_first:
#         # clean span & DOI before returning
#         da_first["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(da_first["in_text_span"], regs))
#         return da_first

#     # 0) Candidate pages: tokens (incl. zenodo record) OR link annotations
#     candidate_pages = []
#     for i, ptxt in enumerate(pages):
#         ok, src = page_has_candidate(info, ptxt, uris_by_page.get(i,[]))
#         if ok or src:
#             candidate_pages.append((i, src or "text"))

#     # 1) Try candidate pages first — FOOTNOTE chain has top priority
#     for i, src in candidate_pages:
#         page_text = pages[i]

#         # 1a) Footnote → main-body anchor
#         fnum, fline = get_footnote_entry_for_dataset(page_text, info)
#         if fnum is not None:
#             anchor_span, anchor_line = find_reference_sentence_for_marker(page_text, fnum)
#             if anchor_span:
#                 dataset_in_paper = ""
#                 def extract_any(clean_blob: str) -> str:
#                     cb = collapse_hyphen_breaks(clean_blob or "")
#                     for R in regs:
#                         mm = R.search(cb)
#                         if mm:
#                             g = mm.group(1) if mm.groups() else mm.group(0)
#                             d = normalize_captured_doi(g)
#                             if d.startswith("10."): return d
#                     return ""
#                 dataset_in_paper = extract_any(fline) or extract_any(anchor_span) or (info.base_doi or "")

#                 lines_all = text_to_lines(full_text); pos_in_full = full_text.find(anchor_span[:60])
#                 section_guess = guess_section_from_lines(lines_all, full_text, pos_in_full) if pos_in_full>=0 else ""
#                 type_code, type_label = classify_source_type(anchor_line or "", section_guess, True)

#                 arrow = f"Footnote {fnum}: {fline}\n    → Anchor: {anchor_span}"
#                 result.update({
#                     "in_text_span": anchor_span, "anchor_sentence": anchor_line or "",
#                     "footnote_number": str(fnum), "footnote_text": fline, "page_index": str(i),
#                     "match_label": f"footnote_superscript_{src}", "match_confidence": 0.97,
#                     "dataset_in_paper": dataset_in_paper, "relation_hint": "availability/archival",
#                     "span_source": f"page_{i}", "arrow_chain": arrow, "debug": "",
#                     "section_guess": section_guess, "source_type": type_code, "source_type_label": type_label
#                 })
#                 result["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(result["in_text_span"], regs))
#                 return result

#         # 1b) Plain text on page (DOI or repo URL/record)
#         sent_hit = sentence_containing_id(page_text, info)
#         if sent_hit and src == "text":
#             s_start, s_end, sent, matched = sent_hit
#             # expand to block if a heading is nearby
#             b_start, b_end = block_span_from_headings(page_text, s_start, s_end)
#             clean_page = collapse_hyphen_breaks(page_text)
#             span = clean_page[b_start:b_end].strip()

#             dataset_in_paper = ""
#             if matched:
#                 mclean = normalize_captured_doi(matched)
#                 dataset_in_paper = mclean if mclean.startswith("10.") else (info.base_doi or "")

#             lines_all = text_to_lines(full_text); snippet = span[:60]
#             pos_in_full = full_text.find(snippet)
#             section_guess = guess_section_from_lines(lines_all, full_text, pos_in_full) if pos_in_full>=0 else ""
#             type_code, type_label = classify_source_type(sent, section_guess, False)

#             arrow = f"Footnote: (none)\n    → Anchor: {span}"
#             result.update({
#                 "in_text_span": span, "anchor_sentence": sent, "footnote_number": "",
#                 "footnote_text": "", "page_index": str(i), "match_label": "text_body_id",
#                 "match_confidence": 0.94, "dataset_in_paper": dataset_in_paper,
#                 "relation_hint": "availability/archival", "span_source": f"page_{i}",
#                 "arrow_chain": arrow, "debug": "plain_text_with_block",
#                 "section_guess": section_guess, "source_type": type_code, "source_type_label": type_label
#             })
#             result["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(result["in_text_span"], regs))
#             return result

#         # 1c) Link-only page: synthesize footnote and pick best body line
#         if src == "link":
#             main_lines, _ = split_main_vs_footnote_lines(page_text)
#             if not main_lines: main_lines = [ln for ln in page_text.splitlines() if ln.strip()]
#             best, best_score = "", 0
#             kw = ["dataset","data","release","available","download","repository","zenodo","dryad","figshare","tcia"]
#             for ln in main_lines:
#                 low = ln.lower(); score = sum(k in low for k in kw) + (1 if 50 <= len(ln) <= 250 else 0)
#                 if score > best_score: best, best_score = ln, score
#             page_join = "\n".join(main_lines)
#             pos = page_join.find(best[:80]) if best else 0
#             if pos < 0: pos = max(0, page_join.lower().find(best.strip().lower()[:40]))
#             s0, e0, exact, _, _ = find_sentence_span(page_join, max(0,pos), CTX_SENT_WIN)
#             _, _, anchor_span = expand_small_span(page_join, s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)

#             lines_all = text_to_lines(full_text); pos_in_full = full_text.find(anchor_span[:60])
#             section_guess = guess_section_from_lines(lines_all, full_text, pos_in_full) if pos_in_full>=0 else ""
#             type_code, type_label = classify_source_type(best or "", section_guess, False)

#             synth_foot = f"Dataset link: {info.base_doi or info.raw}"
#             arrow = f"Footnote ?: {synth_foot}\n    → Anchor: {anchor_span}"
#             result.update({
#                 "in_text_span":anchor_span,"anchor_sentence":best or "","footnote_number":"",
#                 "footnote_text":synth_foot,"page_index":str(i),"match_label":"footnote_link_bodyline",
#                 "match_confidence":0.92,"dataset_in_paper":info.base_doi or "","relation_hint":"availability/archival",
#                 "span_source":f"page_{i}","arrow_chain":arrow,"debug":"no_numbered_footnote",
#                 "section_guess": section_guess, "source_type": type_code, "source_type_label": type_label
#             })
#             result["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(result["in_text_span"], regs))
#             return result

#     # 2) Fallback — look for a global Data Availability block that contains the id
#     m = re.search(
#         r"(?i)Data\W*Availability(?:\W*Statement)?|Availability\W*of\W*Data(?:\W*and\W*Materials)?|"
#         r"Data\W*Accessibility|Data\W*and\W*materials\W*availability",
#         full_text
#     )
#     if m:
#         start = m.start(); end = min(len(full_text), start + 10000)
#         da = collapse_hyphen_breaks(full_text[start:end])
#         tok_in = any(tok in canon(da) for tok in candidate_canon_tokens(info))
#         reg_in = any(R.search(da) for R in regs)
#         if tok_in or reg_in:
#             s0, e0, exact, _, _ = find_sentence_span(full_text, start, CTX_SENT_WIN)
#             _, _, span = expand_small_span(full_text, s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)
#             type_code, type_label = classify_source_type(span, "Data Availability", False)
#             arrow = f"Footnote: (none)\n    → Anchor: {span}"
#             result.update({
#                 "in_text_span": span, "anchor_sentence": exact, "match_label":"da_hit",
#                 "match_confidence": 0.95, "span_source":"fulltext",
#                 "arrow_chain": arrow, "debug":"data_availability_block",
#                 "section_guess":"Data Availability", "source_type": type_code, "source_type_label": type_label
#             })
#             result["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(result["in_text_span"], regs))
#             return result

#     # 3) Final fallback — anywhere in full text via loose DOI/URL (trim References)
#     clean_full = collapse_hyphen_breaks(_truncate_before_references(full_text))
#     for R in regs:
#         mm = R.search(clean_full)
#         if mm:
#             pos = mm.start()
#             s0, e0, exact, _, _ = find_sentence_span(clean_full, pos, CTX_SENT_WIN)
#             _, _, span = expand_small_span(clean_full, s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)
#             type_code, type_label = classify_source_type(exact, "", False)
#             arrow = f"Footnote: (none)\n    → Anchor: {span}"
#             result.update({
#                 "in_text_span": span, "anchor_sentence": exact, "match_label":"text_hit",
#                 "match_confidence":0.90, "span_source":"fulltext", "arrow_chain":arrow,
#                 "source_type": type_code, "source_type_label": type_label
#             })
#             result["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(result["in_text_span"], regs))
#             return result

#     return result  # NOT FOUND

# # -------------------- driver --------------------
# def process_one_article(article_id: str, pdf_folder: Path, rows_for_article: List[Dict]) -> List[Dict]:
#     pdf_path = find_pdf_for_article(article_id, pdf_folder)
#     if not pdf_path:
#         return [{
#             "article_id":article_id,"dataset_id":str(r["dataset_id"]),"in_text_span":"PDF NOT FOUND",
#             "anchor_sentence":"","footnote_number":"","footnote_text":"","page_index":"",
#             "section_guess":"","match_label":"","match_confidence":0.0,"repo_guess":"",
#             "dataset_in_paper":"","version_mismatch":"","relation_hint":"unknown","span_source":"",
#             "arrow_chain":"","debug":"pdf_missing","source_type":"","source_type_label":""
#         } for r in rows_for_article]

#     # New: precompute both page renderings for DA-first attempts
#     pages_pypdf2   = _pages_via_pypdf2(str(pdf_path))
#     pages_pdfminer = _pages_via_pdfminer(str(pdf_path))

#     # Maintain your best-extractor pick for the rest of the strategy
#     full_text, pages = extract_text_with_pages(str(pdf_path))
#     if not any(p.strip() for p in pages):
#         return [{
#             "article_id":article_id,"dataset_id":str(r["dataset_id"]),"in_text_span":"EXTRACTION FAILED",
#             "anchor_sentence":"","footnote_number":"","footnote_text":"","page_index":"",
#             "section_guess":"","match_label":"","match_confidence":0.0,"repo_guess":"",
#             "dataset_in_paper":"","version_mismatch":"","relation_hint":"unknown","span_source":"",
#             "arrow_chain":"","debug":"no_text","source_type":"","source_type_label":""
#         } for r in rows_for_article]

#     uris_by_page = extract_link_uris_by_page(str(pdf_path))
#     lines_all = text_to_lines(full_text)

#     out_rows = []
#     for r in rows_for_article:
#         info = parse_dataset_id(str(r["dataset_id"]).strip())
#         regs = doi_regex_list_for(info)

#         # NEW: Try DA block on both engines first; take the first that matches this dataset
#         da_got = None
#         for _pages in (pages_pypdf2, pages_pdfminer):
#             if _pages:
#                 da_hit = _match_in_da_blocks(_pages, info)
#                 if da_hit:
#                     da_hit["in_text_span"] = normalize_ws_for_output(_normalize_doi_in_span(da_hit["in_text_span"], regs))
#                     da_got = da_hit
#                     break
#         if da_got:
#             out_rows.append({
#                 "article_id": article_id,
#                 "dataset_id": r["dataset_id"],
#                 "in_text_span": da_got.get("in_text_span",""),
#                 "anchor_sentence": da_got.get("anchor_sentence",""),
#                 "footnote_number": da_got.get("footnote_number",""),
#                 "footnote_text": da_got.get("footnote_text",""),
#                 "arrow_chain": da_got.get("arrow_chain",""),
#                 "page_index": da_got.get("page_index",""),
#                 "section_guess": da_got.get("section_guess",""),
#                 "match_label": da_got.get("match_label",""),
#                 "match_confidence": float(da_got.get("match_confidence") or 0.0),
#                 "repo_guess": info.repo_hint or "",
#                 "dataset_in_paper": da_got.get("dataset_in_paper",""),
#                 "version_mismatch": "",
#                 "relation_hint": da_got.get("relation_hint","unknown"),
#                 "span_source": da_got.get("span_source",""),
#                 "debug": da_got.get("debug",""),
#                 "source_type": da_got.get("source_type",""),
#                 "source_type_label": da_got.get("source_type_label","")
#             })
#             continue  # next dataset row

#         # Otherwise, use your original strategy (which already DA-firsts on 'pages')
#         m = match_dataset(full_text, pages, uris_by_page, info)

#         version_mismatch = ""
#         if info.base_doi and m.get("dataset_in_paper"):
#             pb = re.sub(r"(\.v\d+)$","", m["dataset_in_paper"])
#             if pb == info.base_doi and info.version and m["dataset_in_paper"] != info.base_doi + info.version:
#                 version_mismatch = f"requested={info.base_doi}{info.version}, paper={m['dataset_in_paper']}"

#         section_guess = m.get("section_guess","")
#         if not section_guess and m.get("in_text_span") not in {"NOT FOUND","EXTRACTION FAILED","PDF NOT FOUND",""}:
#             pos = full_text.find(m["in_text_span"][:60])
#             if pos >= 0:
#                 section_guess = guess_section_from_lines(lines_all, full_text, pos)

#         out_rows.append({
#             "article_id": article_id,
#             "dataset_id": r["dataset_id"],
#             "in_text_span": m.get("in_text_span","NOT FOUND"),
#             "anchor_sentence": m.get("anchor_sentence",""),
#             "footnote_number": m.get("footnote_number",""),
#             "footnote_text": m.get("footnote_text",""),
#             "arrow_chain": m.get("arrow_chain",""),
#             "page_index": m.get("page_index",""),
#             "section_guess": section_guess,
#             "match_label": m.get("match_label",""),
#             "match_confidence": float(m.get("match_confidence") or 0.0),
#             "repo_guess": info.repo_hint or "",
#             "dataset_in_paper": m.get("dataset_in_paper",""),
#             "version_mismatch": version_mismatch,
#             "relation_hint": m.get("relation_hint","unknown"),
#             "span_source": m.get("span_source",""),
#             "debug": m.get("debug",""),
#             "source_type": m.get("source_type",""),
#             "source_type_label": m.get("source_type_label","")
#         })
#     return out_rows

# def run_pipeline(csv_path: str, pdf_folder: Path, debug_ids: List[str]) -> pd.DataFrame:
#     df = pd.read_csv(csv_path)
#     df["article_id"] = df["article_id"].astype(str).str.strip()
#     df["dataset_id"] = df["dataset_id"].astype(str).str.strip()
#     if debug_ids: df = df[df["article_id"].isin(debug_ids)].copy()

#     df["_row_id"] = range(len(df))
#     groups = []
#     for aid, sub in df.groupby("article_id", sort=False):
#         rows = sub[["dataset_id"]].to_dict(orient="records")
#         groups.append((aid, rows))

#     all_res = []
#     with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
#         futs = {ex.submit(process_one_article, aid, pdf_folder, rows): (aid, len(rows)) for aid, rows in groups}
#         for fut in tqdm(as_completed(futs), total=len(futs), desc="Parsing PDFs"):
#             try:
#                 all_res.extend(fut.result())
#             except Exception as e:
#                 aid, nrows = futs[fut]
#                 all_res.extend([{
#                     "article_id":aid,"dataset_id":"","in_text_span":f"ERROR: {e}","anchor_sentence":"",
#                     "footnote_number":"","footnote_text":"","arrow_chain":"","page_index":"",
#                     "section_guess":"","match_label":"","match_confidence":0.0,"repo_guess":"",
#                     "dataset_in_paper":"","version_mismatch":"","relation_hint":"unknown","span_source":"",
#                     "debug":"exception","source_type":"","source_type_label":""
#                 } for _ in range(nrows)])

#     out = pd.DataFrame(all_res)
#     out = df.merge(out, on=["article_id","dataset_id"], how="left").sort_values("_row_id").drop(columns=["_row_id"])

#     for col, default in {
#         "in_text_span":"NOT FOUND","anchor_sentence":"","footnote_number":"","footnote_text":"",
#         "arrow_chain":"","page_index":"","section_guess":"","match_label":"","match_confidence":0.0,
#         "repo_guess":"","dataset_in_paper":"","version_mismatch":"","relation_hint":"unknown",
#         "span_source":"","debug":"","source_type":"","source_type_label":""
#     }.items():
#         out[col] = out[col].fillna(default)

#     mask = (~out["in_text_span"].isin(["NOT FOUND","PDF NOT FOUND","EXTRACTION FAILED"])) & (out["in_text_span"]!="")
#     print(f"Processed {len(groups)} PDFs, {len(df)} rows. Found spans for {int(mask.sum())} rows.")
#     out.to_csv(SAVE_AS, index=False)
#     print(f"Saved: {SAVE_AS}")
#     return out


# out_df = run_pipeline(CSV_PATH, PDF_FOLDER, DEBUG_ARTICLE_IDS)
# out_df.head(12)


In [ ]:
# Run this once per runtime to speed up text extraction.
!apt-get -yqq install poppler-utils

Selecting previously unselected package poppler-utils.
(Reading database ... 126374 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.10_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.10) ...
Setting up poppler-utils (22.02.0-2ubuntu0.10) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import os

# Use almost all cores
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)

# Optional: reduce thread storms inside libs used by pdf parsing
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

'1'

In [ ]:
# =================== Combined extractor: Ref#-first, v3.21 fallback ===================
# EXTREMELY IMPORTANT: Produces "in_text_span" column.
# Order:
#   (A) DOI -> exact ref number -> find paragraphs that truly cite ref via superscript/[n]/guarded ".22,"
#   (B) If (A) fails, run the v3.21 general pipeline to mine Data Availability / inline DOI mentions, etc.

# ─────────────────────────────────────────────────────────────────────────────────────
# Minimal installs (quiet)
import sys, subprocess, shlex, os
def _pip_install(pkg):
    try:
        __import__(pkg.split("==")[0].replace("-", "_"))
    except Exception:
        subprocess.check_call(shlex.split(f"{sys.executable} -m pip install -q {pkg}"))
_pip_install("pymupdf")
_pip_install("pdfminer.six")
_pip_install("pypdf2")
_pip_install("tqdm")
_pip_install("pandas")

# ─────────────────────────────────────────────────────────────────────────────────────
import re, unicodedata, logging, shutil, statistics
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from statistics import median

import fitz  # PyMuPDF
import pandas as pd
from tqdm.auto import tqdm
from PyPDF2 import PdfReader
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer, LAParams

logging.getLogger("PyPDF2").setLevel(logging.ERROR)

# ---- paths (adjust as needed) ----
PDF_FOLDER = Path("/content/drive/MyDrive/Make_data_count_challenge/Data/train/PDF")
CSV_PATH   = "/content/drive/MyDrive/Make_data_count_challenge/Data/train_labels_cleaned.csv"

# For quick testing, keep your debug AID here; otherwise leave [] to process all
DEBUG_ARTICLE_IDS: List[str] = []
SAVE_AS          = "in_text_spans_combined.csv"
MAX_WORKERS      = max(1, (os.cpu_count() or 2) - 1)
RECURSIVE_PDF    = True

# v3.21 tuning
CTX_SENT_WIN     = 1
MIN_SPAN_CHARS   = 180
MAX_SPAN_CHARS   = 1200

# ─────────────────────────────────────────────────────────────────────────────────────
# Shared helpers (canon, paths, DOI normalization, etc.)

def canon(s: str) -> str:
    if s is None: return ""
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s.lower() if ch.isalnum())

def collapse_hyphen_breaks(s: str) -> str:
    s = s.replace("\r", "")
    s = re.sub(r"-\s*\n\s*", "", s)
    s = re.sub(r"(\b10)\s*\.\s*([0-9]{4,9})\s*/\s*", r"\1.\2/", s, flags=re.I)
    s = re.sub(r"(?i)\bdoi\s*:\s*", "", s)
    s = re.sub(r"(?i)doi\s*\.\s*org", "doi.org", s)
    s = re.sub(r"(?i)dx\s*\.\s*doi\s*\.\s*org", "dx.doi.org", s)
    s = re.sub(r"(?i)https?\s*:\s*/\s*/", "https://", s)
    return s


def normalize_ws_for_output(s: str) -> str:
    """
    Stage 2: presentation
      - convert single newlines to spaces (paragraph reflow)
      - keep blank lines as paragraph breaks
      - collapse multiple spaces, strip
    """
    # Protect paragraph breaks first
    s = re.sub(r"\n\s*\n", "¶¶", s)        # mark blank-line paragraph breaks
    s = s.replace("\n", " ")               # single LF => space
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace("¶¶", "\n\n")            # restore paragraph breaks
    return s

def find_pdf_for_article(article_id: str, pdf_folder: Path) -> Optional[Path]:
    direct = pdf_folder / f"{article_id}.pdf"
    if direct.exists():
        return direct
    variants = {article_id, article_id.replace("/", "_"), article_id.replace("_", "/")}
    it = pdf_folder.rglob("*.pdf") if RECURSIVE_PDF else pdf_folder.glob("*.pdf")
    for f in it:
        if any(v in f.stem for v in variants):
            return f
    return None

# ─────────────────────────────────────────────────────────────────────────────────────
# (A) DOI -> reference number -> superscript/[n]/guarded extractor (PyMuPDF span-aware)

DOI_RX = re.compile(r"(10\.\d{4,9}/[-._;()/:A-Z0-9]+)", re.I)

def _norm_doi(s: str) -> str:
    s = (s or "").strip().lower().replace("\u200b","").replace("\ufeff","").replace("\u00ad","")
    s = s.replace("https://doi.org/","").replace("http://doi.org/","")
    return re.sub(r"\s+","", s)

def _refs_start_page(doc: fitz.Document) -> Optional[int]:
    rx = re.compile(r"\bREFERENCES\b", re.I)
    for pno in range(len(doc)-1, -1, -1):
        if rx.search(doc[pno].get_text("text")):
            return pno
    return None

def _parse_references_strict(doc: fitz.Document, start_page: int) -> Dict[str, str]:
    lines = []
    for p in range(start_page, len(doc)):
        lines += doc[p].get_text("text").replace("\r","\n").split("\n")
    refs, cur, buf = {}, None, []
    start_rx = re.compile(r"^\s*(\d{1,3})\.\s+(.*)$")
    def flush():
        nonlocal cur, buf
        if cur and buf:
            refs[cur] = " ".join(x.strip() for x in buf if x.strip())
        cur, buf = None, []
    for ln in lines:
        m = start_rx.match(ln)
        if m:
            flush(); cur = m.group(1); buf = [m.group(2)]
        elif cur:
            buf.append(ln)
    flush()
    return refs

def _doi_to_refnum_exact(ref_map: Dict[str,str], doi_raw: str) -> Optional[str]:
    tgt = _norm_doi(doi_raw)
    for num, entry in ref_map.items():
        for d in DOI_RX.findall(entry):
            if _norm_doi(d) == tgt:
                return num
    return None

def _page_paras_with_lines_spans(page: fitz.Page, gap_factor=0.65):
    d = page.get_text("dict")
    paras = []
    for block in d.get("blocks", []):
        if block.get("type", 0) != 0: continue
        raw = []
        for ln in block.get("lines", []):
            spans = ln.get("spans", [])
            if not spans: continue
            y0 = min(sp["bbox"][1] for sp in spans if "bbox" in sp)
            y1 = max(sp["bbox"][3] for sp in spans if "bbox" in sp)
            line_text = "".join(sp.get("text","") for sp in spans).replace("\r","")
            line_spans = [{"text": sp.get("text",""), "size": sp.get("size",0), "bbox": sp.get("bbox",[0,0,0,0])} for sp in spans]
            raw.append((y0,y1,line_text,line_spans))
        if not raw: continue
        heights = [y1-y0 for (y0,y1,_,_) in raw]
        med_h = sorted(heights)[len(heights)//2] or 1.0
        cur_lines, cur_texts, prev = [], [], None
        for (y0,y1,ltxt,lspans) in raw:
            if prev is None:
                cur_lines, cur_texts = [ {"text": ltxt, "spans": lspans} ], [ltxt]
            else:
                gap = y0 - prev
                if gap > gap_factor * med_h:
                    paras.append({"text":"\n".join(cur_texts), "lines":cur_lines})
                    cur_lines, cur_texts = [ {"text": ltxt, "spans": lspans} ], [ltxt]
                else:
                    cur_lines.append({"text": ltxt, "spans": lspans})
                    cur_texts.append(ltxt)
            prev = y1
        if cur_lines:
            paras.append({"text":"\n".join(cur_texts), "lines":cur_lines})
    return [p for p in paras if p["text"].strip()]

def _is_sentence_complete(s: str) -> bool:
    return re.search(r"[.!?][\"\')\]]*\s*$", s.strip()) is not None

def _looks_like_continuation(text: str) -> bool:
    first = next((ln for ln in text.splitlines() if ln.strip()), "")
    if not first: return False
    if re.match(r"^[,.;:\-\)\]]", first): return True
    if re.match(r"^[a-z]", first): return True
    if len(first.split()) < 3: return True
    return False

def _merge_continuations(paras):
    merged, i = [], 0
    while i < len(paras):
        cur = paras[i]
        j = i
        while (j+1 < len(paras)) and (not _is_sentence_complete(cur["text"])) and _looks_like_continuation(paras[j+1]["text"]):
            cur = {"text": cur["text"] + "\n" + paras[j+1]["text"],
                   "lines": cur["lines"] + paras[j+1]["lines"]}
            j += 1
        merged.append(cur)
        i = j + 1
    return merged

# ─── CLEANING (keep + pretty) ────────────────────────────────────────────────
def _clean_keep_lines(s: str) -> str:
    """
    Stage 1: structural cleaning
      - remove soft hyphen / BOM
      - de-hyphenate wrapped words
      - trim trailing spaces per line
    (We keep line breaks here; pretty-join happens in normalize_ws_for_output.)
    """
    s = s.replace("\u200b","").replace("\ufeff","").replace("\u00ad","")
    s = re.sub(r"(\w)-\n(\w)", r"\1\2", s)                 # de-hyphenate wrap
    s = "\n".join(re.sub(r"[ \t]+"," ", ln).rstrip() for ln in s.splitlines())
    return s.strip()

def _line_has_sup_or_bracket_or_guarded(line: Dict, ref_num: str) -> bool:
    spans = line["spans"]
    text  = line["text"]
    # [n]
    if re.search(rf"\[\s*{re.escape(ref_num)}\s*\]", text): return True
    # superscript via smaller span
    if spans:
        sizes = [sp.get("size", 0) for sp in spans if sp.get("size",0) > 0]
        med_size = median(sizes) if sizes else 0
        small_thresh = med_size * 0.92 if med_size else 0
        for i, sp in enumerate(spans):
            if sp.get("text","").strip() == ref_num:
                size = sp.get("size", 0)
                if med_size and size and size < small_thresh:
                    prev_text = spans[i-1].get("text","") if i>0 else ""
                    prev_char = prev_text.rstrip()[-1:] if prev_text else ""
                    if prev_char in {".",")","]"} or prev_char == "":
                        return True
    # punctuation-guarded textual fallback, not decimals
    if re.search(rf"(?<!\d)[\.\)\]]\s*{re.escape(ref_num)}(?=[,;])", text):
        return True
    return False

def _para_has_true_ref(para: Dict, ref_num: str):
    for idx, line in enumerate(para["lines"], start=1):
        if _line_has_sup_or_bracket_or_guarded(line, ref_num):
            return True, line["text"], idx
    return False, "", None

# ─── REF# → IN-TEXT PARAGRAPHS (pretty formatting) ───────────────────────────
def extract_intext_by_refnum_first(pdf_path: Path, dataset_id: str) -> Optional[str]:
    """
    Returns a SINGLE pretty string that may contain multiple paragraphs, each followed by a
    crisp locator line. Formatting matches the fallback style (no stray \n inside sentences).
    """
    try:
        doc = fitz.open(str(pdf_path))
    except Exception:
        return None

    refs_start = _refs_start_page(doc)
    if refs_start is None:
        return None
    ref_map = _parse_references_strict(doc, refs_start)
    ref_num = _doi_to_refnum_exact(ref_map, dataset_id)
    if not ref_num:
        return None

    collected = []
    for pno in range(refs_start):  # body pages only
        raw_paras = _page_paras_with_lines_spans(doc[pno])
        merged_paras = _merge_continuations(raw_paras)
        for idx, para in enumerate(merged_paras, start=1):
            hit, loc_line, loc_idx = _para_has_true_ref(para, ref_num)
            if not hit:
                continue

            # Clean structure, then pretty-print for output
            cleaned_par  = _clean_keep_lines(para["text"])
            pretty_par   = normalize_ws_for_output(cleaned_par)

            cleaned_line = _clean_keep_lines(loc_line) if loc_line else ""
            pretty_line  = normalize_ws_for_output(cleaned_line)

            # Locator with consistent phrasing
            locator = (
                f"{dataset_id} which is reference number {ref_num} indicated at this line "
                f"{pretty_line} of paragraph {idx}"
                if loc_idx is not None else
                f"{dataset_id} which is reference number {ref_num} indicated in paragraph {idx}"
            )

            collected.append(pretty_par + "\n\n" + locator)

    return ("\n\n---\n\n".join(collected)) if collected else None

# ─────────────────────────────────────────────────────────────────────────────────────
# (B) v3.21 fallback (unchanged logic, kept compact but functionally identical where needed)

def _try_pdftotext(pdf_path: str):
    if not shutil.which("pdftotext"):
        return "", []
    r = subprocess.run(["pdftotext", "-layout", pdf_path, "-"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False)
    if not r.stdout: return "", []
    txt = r.stdout
    pages = txt.split("\f") if "\f" in txt else [txt]
    return "\n".join(pages), pages

def _text_quality_metrics(text: str) -> dict:
    letters = sum(ch.isalpha() for ch in text)
    digits  = sum(ch.isdigit() for ch in text)
    spaces  = text.count(" ")
    tokens  = text.split()
    avg_tok = statistics.mean([len(w) for w in tokens]) if tokens else 0.0
    long_tok = sum(1 for w in tokens if len(w) >= 20)
    l2d = len(re.findall(r"[A-Za-z][0-9]", text))
    return {"letters":letters,"digits":digits,"spaces":spaces,"space_ratio": spaces/max(1,letters+digits),
            "avg_token_len":avg_tok,"long_tokens":long_tok,"l2d":l2d,"nonempty": any(ln.strip() for ln in text.splitlines())}

def _score_pages(pages: List[str]) -> float:
    if not pages: return -1e9
    scores = []
    for t in pages:
        m = _text_quality_metrics(t)
        if not m["nonempty"]: scores.append(-1e9); continue
        s = (100.0*m["space_ratio"] - 1.5*m["avg_token_len"]
             - 10.0*(m["long_tokens"]/max(1,len(t.split()))) - 0.2*m["l2d"])
        scores.append(s)
    return sum(scores)/len(scores)

def _try_pdfminer(pdf_path: str):
    lap = LAParams(char_margin=2.0, word_margin=0.15, line_margin=0.5, boxes_flow=0.5, all_texts=True, detect_vertical=False)
    pages = []
    try:
        for layout in extract_pages(pdf_path, laparams=lap):
            buf = []
            for el in layout:
                if isinstance(el, LTTextContainer):
                    buf.append(el.get_text())
            pages.append("".join(buf))
    except Exception:
        return "", []
    return "\n".join(pages), pages

def _try_pypdf2(pdf_path: str):
    try:
        reader = PdfReader(pdf_path, strict=False)
        pages = []
        for page in reader.pages:
            try:
                t = page.extract_text()
            except Exception:
                t = None
            pages.append(t or "")
        return ("\n".join(pages), pages) if any(p.strip() for p in pages) else ("", [])
    except Exception:
        return "", []

def extract_text_with_pages(pdf_path: str):
    cands = []
    full, pages = _try_pdftotext(pdf_path)
    if pages: cands.append(("pdftotext", full, pages, _score_pages(pages)))
    full, pages = _try_pdfminer(pdf_path)
    if pages: cands.append(("pdfminer", full, pages, _score_pages(pages)))
    full, pages = _try_pypdf2(pdf_path)
    if pages: cands.append(("pypdf2", full, pages, _score_pages(pages)))
    if not cands: return "", []
    best = max(cands, key=lambda x: x[3])
    return best[1], best[2]

# --- DA blocks (same as v3.21, trimmed to essentials)
def unsplit_spaced_caps(line: str) -> str:
    return re.sub(r'(?<![A-Za-z])(?:[A-Z]\s+){2,}[A-Z](?![A-Za-z])', lambda m: m.group(0).replace(' ', ''), line)

DA_HEAD_PATTERNS = [
    r"(?i)^\s*Data\W*Availability(?:\W*Statement)?\s*$",
    r"(?i)^\s*Availability\W*of\W*Data(?:\W*and\W*Materials)?\s*$",
    r"(?i)^\s*Data\W*Accessibility\s*$",
    r"(?i)^\s*Data\W*and\W*materials\W*availability\s*$",
    r"(?i)^\s*(?:■\s*)?ASSOCIATED\W*CONTENT\s*$",
    r"(?i)^\s*SUPPORTING\W*INFORMATION\s*$",
]

def _find_section_blocks_on_page(page_text: str):
    page_clean = collapse_hyphen_breaks(page_text)
    lines = page_clean.splitlines(True)
    idxs = []
    for i, ln in enumerate(lines):
        chk = unsplit_spaced_caps(ln)
        for pat in DA_HEAD_PATTERNS:
            if re.match(pat, chk.strip()):
                j = i + 1
                while j < len(lines):
                    L = unsplit_spaced_caps(lines[j]).strip()
                    if not L and j+1 < len(lines) and not lines[j+1].strip(): break
                    next_chunk = "".join(lines[j:j+3])
                    if (len(L) <= 80 and (L.isupper() or re.match(
                        r"(?i)^\s*(references|acknowledg|author information|notes|conclusions|appendix|methods|materials|results|discussion|orcid)\b", L))
                        and not re.search(r"(?:doi\s*:?\s*)?(?:https?://)?doi\.org/10\.\d{4,9}/|10\.\d{4,9}/|dryad|figshare|zenodo", next_chunk, re.I)):
                        break
                    j += 1
                start = sum(len(x) for x in lines[:i]); end = sum(len(x) for x in lines[:j])
                idxs.append((start, end, chk.strip()))
                break
    return idxs

@dataclass
class DatasetIdInfo:
    raw: str
    lower: str
    base_doi: Optional[str]
    version: Optional[str]
    repo_hint: Optional[str]
    canon_target: str
    canon_suffix: str
    zenodo_id: Optional[str]

def _infer_repo(lower_id: str) -> Optional[str]:
    if "dryad" in lower_id or "10.5061" in lower_id: return "Dryad"
    if "tcia" in lower_id or "10.7937" in lower_id: return "TCIA"
    if "mendeley" in lower_id or "10.17632" in lower_id: return "Mendeley Data"
    if "zenodo" in lower_id or "10.5281" in lower_id: return "Zenodo"
    return None

def parse_dataset_id(dataset_id: str) -> DatasetIdInfo:
    s = str(dataset_id).strip(); lower = s.lower(); doi = None
    if lower.startswith(("http://","https://")):
        lower_no_proto = re.sub(r"^https?://", "", lower)
        lower_no_proto = re.sub(r"^(dx\.)?doi\.org/", "", lower_no_proto)
        m = re.search(r"(10\.\d{4,9}/[^\s]+)", lower_no_proto)
        if m: doi = m.group(1)
    elif lower.startswith("10."):
        doi = lower
    base_doi, version = None, None
    if doi:
        doi = doi.strip().strip(".,;")
        m = re.match(r"^(10\.[^ ]+?)(\.v\d+)$", doi)
        if m: base_doi, version = m.group(1), m.group(2)
        else: base_doi = doi
    repo_hint = _infer_repo(lower if not doi else doi)
    suffix = base_doi.split("/",1)[-1] if base_doi else lower.split("/",1)[-1]
    zenodo_id = None
    if (base_doi or "").startswith("10.5281/zenodo."):
        zenodo_id = (base_doi or lower).split("zenodo.",1)[-1].split("/",1)[0].split(".v",1)[0]
    return DatasetIdInfo(
        raw=s, lower=lower, base_doi=base_doi, version=version, repo_hint=repo_hint,
        canon_target=canon(base_doi or lower), canon_suffix=canon(suffix or lower), zenodo_id=zenodo_id
    )

def _loose_tail_regex(tail: str) -> str:
    pat = []
    for ch in tail:
        if ch.isalnum(): pat.append(f"{re.escape(ch)}\\s*")
        elif ch in "./:_-;": pat.append(f"\\s*{re.escape(ch)}\\s*")
        else: pat.append(f"\\s*{re.escape(ch)}\\s*")
    return "".join(pat)

def doi_regex_list_for(info: DatasetIdInfo) -> List[re.Pattern]:
    regs = []
    if info.base_doi:
        tail = info.base_doi.split("/",1)[-1]
        tail_loose = _loose_tail_regex(tail)
        regs.append(re.compile(
            r"(?:doi\s*:?\s*)?"
            r"(?:https?://(?:dx\.)?doi\.org/\s*)?"
            r"(10\s*\.\s*[0-9\s]{4,9}\s*/\s*" + tail_loose + r")"
            r"(?:\s*\.v\d+)?",
            flags=re.IGNORECASE | re.DOTALL
        ))
    return regs

def extract_link_uris_by_page(pdf_path: str) -> Dict[int, List[str]]:
    uris_by_page = {}
    try:
        reader = PdfReader(pdf_path, strict=False)
        for i, page in enumerate(reader.pages):
            uris = []
            annots = page.get("/Annots")
            if annots:
                for a in annots:
                    try:
                        obj = a.get_object()
                        if "/A" in obj and "/URI" in obj["/A"]:
                            uris.append(str(obj["/A"]["/URI"]))
                        if "/URI" in obj:
                            uris.append(str(obj["/URI"]))
                    except Exception:
                        continue
            uris_by_page[i] = uris
    except Exception:
        pass
    return uris_by_page

def text_to_lines(text: str) -> List[str]:
    return text.splitlines()

def split_sentences(text: str) -> List[str]:
    t = re.sub(r"[ \t]+", " ", text)
    for a in ["et al.","e.g.","i.e.","Dr.","Prof.","Fig.","Eq.","Ref.","Refs.","No.","Vol.","pp.","Inc.","Ltd."]:
        t = t.replace(a, a.replace(".", "◊"))
    parts = re.split(r"(?<=[\.\?\!])\s+(?=[A-Z0-9\[])|(?<=\.)\n+", t)
    return [p.replace("◊",".").strip() for p in parts if p and p.strip()]

def find_sentence_span(text: str, match_start: int, ctx_sent_win: int = 1) -> Tuple[int,int,str,str,str]:
    sents = split_sentences(text)
    spans, off = [], 0
    for sent in sents:
        idx = text.find(sent, off)
        if idx < 0:
            idx = text[off:].find(sent)
            if idx >= 0: idx += off
        if idx >= 0:
            spans.append((idx, idx+len(sent))); off = idx+len(sent)
        else:
            spans.append((off, off+len(sent))); off += len(sent)
    si = 0
    for i,(a,b) in enumerate(spans):
        if a <= match_start < b:
            si = i; break
    li = max(0, si-ctx_sent_win); ri = min(len(sents)-1, si+ctx_sent_win)
    left  = " ".join(sents[li:si]) if si>li else ""
    exact = sents[si]
    right = " ".join(sents[si+1:ri+1]) if ri>si else ""
    return spans[si][0], spans[si][1], exact.strip(), left.strip(), right.strip()

def expand_small_span(text: str, start: int, end: int, min_chars: int, max_chars: int) -> Tuple[int,int,str]:
    if end - start >= min_chars:
        return start, end, text[start:end].strip()
    pL = text.rfind("\n\n", 0, start); pR = text.find("\n\n", end)
    if pL == -1: pL = 0
    else: pL += 2
    if pR == -1: pR = len(text)
    blob = text[pL:pR].strip()
    if len(blob) >= min_chars:
        return pL, pR, (blob[:max_chars].rsplit(" ",1)[0] + " …") if len(blob)>max_chars else blob
    half = max(min_chars//2, 160)
    L = max(0, start-half); R = min(len(text), end+half)
    snip = text[L:R].strip()
    return L, R, (snip[:max_chars].rsplit(" ",1)[0] + " …") if len(snip)>max_chars else snip

# DA-first for a given page list
def _match_in_da_blocks(pages: List[str], info: DatasetIdInfo) -> Optional[Dict]:
    regs = doi_regex_list_for(info)
    for pi, page in enumerate(pages):
        page_clean = collapse_hyphen_breaks(page)
        blocks = _find_section_blocks_on_page(page_clean)
        if not blocks: continue
        for (a,b,head) in blocks:
            block = page_clean[a:b].strip()
            tok_hit = (info.canon_target in canon(block)) or (info.canon_suffix in canon(block))
            reg_hit = any(R.search(block) for R in regs)
            if tok_hit or reg_hit:
                # Canonicalize DOI string in block
                for R in regs:
                    m = R.search(block)
                    if m:
                        g = m.group(1) if m.groups() else m.group(0)
                        d = re.sub(r"\s+","", g).lower()
                        block = block.replace(m.group(0), f"https://doi.org/{d}")
                        break
                return {
                    "in_text_span": normalize_ws_for_output(block),
                    "match_label": "da_hit_priority",
                    "match_confidence": 0.98,
                    "page_index": str(pi),
                    "section_guess": head or "Data Availability",
                }
    return None

def sentence_containing_id(page_text: str, info: DatasetIdInfo):
    clean = collapse_hyphen_breaks(page_text)
    regs = doi_regex_list_for(info)
    for R in regs:
        m = R.search(clean)
        if m:
            hit = (m.group(1) if m.groups() else m.group(0))
            s0,e0, sent, _, _ = find_sentence_span(clean, m.start(), CTX_SENT_WIN)
            return s0,e0,sent,hit
    return None

def classify_source_type(anchor_sentence: str, section_heading: str, has_footnote: bool):
    s = anchor_sentence or ""
    h = section_heading or ""
    s_low = s.lower(); h_low = h.lower()
    if has_footnote: return ("2-3", "Footnote or endnote (superscript)")
    if re.search(r"(?i)\bdata\s+availability\b|\bavailability\s+of\s+data\b|\bdata\s+accessibilit", h) \
       or re.search(r"(?i)\bdata\s+availability\b", s): return ("2-5","Data availability section")
    if re.search(r"\[\d+\]|\(\d+\)", s): return ("2-2","Numeric citation (Vancouver/IEEE)")
    if re.search(r"https?://|doi\.org", s_low): return ("2-4","Inline link (URL or DOI in text)")
    return ("2-16","OTHER WAY")

def _pages_via_pypdf2(pdf_path: str) -> List[str]:
    try:
        reader = PdfReader(pdf_path, strict=False)
        return [(p.extract_text() or "") for p in reader.pages]
    except Exception:
        return []

def _pages_via_pdfminer(pdf_path: str) -> List[str]:
    try:
        lap = LAParams(char_margin=2.0, word_margin=0.15, line_margin=0.5,
                       boxes_flow=0.5, all_texts=True, detect_vertical=False)
        pages = []
        for layout in extract_pages(pdf_path, laparams=lap):
            buf = []
            for el in layout:
                if isinstance(el, LTTextContainer):
                    buf.append(el.get_text())
            pages.append("".join(buf))
        return pages
    except Exception:
        return []

# ─────────────────────────────────────────────────────────────────────────────────────
# Driver: per-article processing with ref#-first, then fallback

def process_one_article(article_id: str, pdf_folder: Path, rows_for_article: List[Dict]) -> List[Dict]:
    pdf_path = find_pdf_for_article(article_id, pdf_folder)
    if not pdf_path:
        return [{
            "article_id":article_id,"dataset_id":str(r["dataset_id"]),
            "in_text_span":"PDF NOT FOUND","anchor_sentence":"","footnote_number":"",
            "footnote_text":"","arrow_chain":"","page_index":"","section_guess":"",
            "match_label":"","match_confidence":0.0,"repo_guess":"","dataset_in_paper":"",
            "version_mismatch":"","relation_hint":"unknown","span_source":"","debug":"",
            "source_type":"","source_type_label":""
        } for r in rows_for_article]

    # Try the ref-number mechanism FIRST for EACH dataset row
    out_rows: List[Dict] = []
    # Also precompute v3.21 materials in case we need fallback
    full_text, pages_best = extract_text_with_pages(str(pdf_path))
    pages_pypdf2   = _pages_via_pypdf2(str(pdf_path))
    pages_pdfminer = _pages_via_pdfminer(str(pdf_path))
    uris_by_page   = extract_link_uris_by_page(str(pdf_path))
    lines_all      = text_to_lines(full_text)

    for r in rows_for_article:
        ds_id = str(r["dataset_id"]).strip()

        # (A) DOI->ref#->superscript/[n]/guarded
        try:
            ref_first_span = extract_intext_by_refnum_first(pdf_path, ds_id)
        except Exception:
            ref_first_span = None

        if ref_first_span:
            out_rows.append({
                "article_id": article_id,
                "dataset_id": ds_id,
                "in_text_span": ref_first_span,
                "anchor_sentence": "",
                "footnote_number": "",
                "footnote_text": "",
                "arrow_chain": "",
                "page_index": "",
                "section_guess": "",
                "match_label": "refnum_superscript_guarded",
                "match_confidence": 0.99,
                "repo_guess": "",
                "dataset_in_paper": "",
                "version_mismatch": "",
                "relation_hint": "availability/archival",
                "span_source": "pymupdf_refnum",
                "debug": "",
                "source_type": "2-2",
                "source_type_label": "Numeric citation (Vancouver/IEEE)"
            })
            continue  # next dataset

        # (B) Fallback: v3.21 flow (DA-first on two engines, then best extractor text)
        info = parse_dataset_id(ds_id)
        regs = doi_regex_list_for(info)

        # DA blocks on both renderings
        picked_da = None
        for _pages in (pages_pypdf2, pages_pdfminer):
            if not _pages: continue
            da = _match_in_da_blocks(_pages, info)
            if da:
                picked_da = da
                break

        if picked_da:
            out_rows.append({
                "article_id": article_id,
                "dataset_id": ds_id,
                "in_text_span": picked_da["in_text_span"],
                "anchor_sentence": "",
                "footnote_number": "",
                "footnote_text": "",
                "arrow_chain": "",
                "page_index": picked_da.get("page_index",""),
                "section_guess": picked_da.get("section_guess","Data Availability"),
                "match_label": picked_da.get("match_label","da_hit_priority"),
                "match_confidence": picked_da.get("match_confidence",0.98),
                "repo_guess": info.repo_hint or "",
                "dataset_in_paper": info.base_doi or "",
                "version_mismatch": "",
                "relation_hint": "availability/archival",
                "span_source": "da_block",
                "debug": "",
                "source_type": "2-5",
                "source_type_label": "Data availability section"
            })
            continue

        # Plain text inline DOI on best pages
        got = False
        for i, page_text in enumerate(pages_best):
            sc = sentence_containing_id(page_text, info)
            if not sc: continue
            s0,e0,sent,matched = sc
            # expand to a compact paragraph-ish block
            b0, b1, span = expand_small_span(collapse_hyphen_breaks(page_text), s0, e0, MIN_SPAN_CHARS, MAX_SPAN_CHARS)
            span = normalize_ws_for_output(span)
            tcode, tlabel = classify_source_type(sent, "", False)
            out_rows.append({
                "article_id": article_id,
                "dataset_id": ds_id,
                "in_text_span": span,
                "anchor_sentence": sent,
                "footnote_number": "",
                "footnote_text": "",
                "arrow_chain": "",
                "page_index": str(i),
                "section_guess": "",
                "match_label": "text_body_id",
                "match_confidence": 0.94,
                "repo_guess": info.repo_hint or "",
                "dataset_in_paper": info.base_doi or "",
                "version_mismatch": "",
                "relation_hint": "availability/archival",
                "span_source": f"page_{i}",
                "debug": "plain_text_with_block",
                "source_type": tcode,
                "source_type_label": tlabel
            })
            got = True
            break
        if got:
            continue

        # Nothing found
        out_rows.append({
            "article_id": article_id,
            "dataset_id": ds_id,
            "in_text_span": "NOT FOUND",
            "anchor_sentence": "",
            "footnote_number": "",
            "footnote_text": "",
            "arrow_chain": "",
            "page_index": "",
            "section_guess": "",
            "match_label": "",
            "match_confidence": 0.0,
            "repo_guess": info.repo_hint or "",
            "dataset_in_paper": "",
            "version_mismatch": "",
            "relation_hint": "unknown",
            "span_source": "",
            "debug": "no_match_any_strategy",
            "source_type": "",
            "source_type_label": ""
        })

    return out_rows

# ─────────────────────────────────────────────────────────────────────────────────────
# Batch runner

def run_pipeline(csv_path: str, pdf_folder: Path, debug_ids: List[str]) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["article_id"] = df["article_id"].astype(str).str.strip()
    df["dataset_id"] = df["dataset_id"].astype(str).str.strip()
    if debug_ids: df = df[df["article_id"].isin(debug_ids)].copy()

    df["_row_id"] = range(len(df))
    groups = []
    for aid, sub in df.groupby("article_id", sort=False):
        rows = sub[["dataset_id"]].to_dict(orient="records")
        groups.append((aid, rows))

    # sequential to keep logs clean in Colab; you can parallelize if you want
    all_res = []
    for aid, rows in tqdm(groups, desc="Parsing PDFs"):
        try:
            all_res.extend(process_one_article(aid, pdf_folder, rows))
        except Exception as e:
            all_res.extend([{
                "article_id":aid,"dataset_id":r["dataset_id"],"in_text_span":f"ERROR: {e}",
                "anchor_sentence":"","footnote_number":"","footnote_text":"","arrow_chain":"",
                "page_index":"","section_guess":"","match_label":"","match_confidence":0.0,
                "repo_guess":"","dataset_in_paper":"","version_mismatch":"","relation_hint":"unknown",
                "span_source":"","debug":"exception","source_type":"","source_type_label":""
            } for r in rows])

    out = pd.DataFrame(all_res)
    out = df.merge(out, on=["article_id","dataset_id"], how="left").sort_values("_row_id").drop(columns=["_row_id"])

    # Fill empties
    defaults = {
        "in_text_span":"NOT FOUND","anchor_sentence":"","footnote_number":"","footnote_text":"",
        "arrow_chain":"","page_index":"","section_guess":"","match_label":"","match_confidence":0.0,
        "repo_guess":"","dataset_in_paper":"","version_mismatch":"","relation_hint":"unknown",
        "span_source":"","debug":"","source_type":"","source_type_label":""
    }
    for col, default in defaults.items():
        out[col] = out[col].fillna(default)

    # Save
    mask = (~out["in_text_span"].isin(["NOT FOUND","PDF NOT FOUND","EXTRACTION FAILED"])) & (out["in_text_span"]!="")
    print(f"Processed {len(groups)} PDFs, {len(df)} rows. Found spans for {int(mask.sum())} rows.")
    out.to_csv(SAVE_AS, index=False)
    print(f"Saved: {SAVE_AS}")
    return out

# ─────────────────────────────────────────────────────────────────────────────────────
# Run
out_df = run_pipeline(CSV_PATH, PDF_FOLDER, DEBUG_ARTICLE_IDS)
out_df.head(12)

Parsing PDFs:   0%|          | 0/213 [00:00<?, ?it/s]

Processed 213 PDFs, 718 rows. Found spans for 315 rows.
Saved: in_text_spans_combined.csv


,article_id,dataset_id,type,in_text_span,anchor_sentence,footnote_number,footnote_text,arrow_chain,page_index,section_guess,match_label,match_confidence,repo_guess,dataset_in_paper,version_mismatch,relation_hint,span_source,debug,source_type,source_type_label
0,10.1002_2017jc013030,https://doi.org/10.17882/49388,Primary,Journal of Geophysical Research: Oceans 10.100...,Data referring to Organelli\net al. (2016a; ht...,,,,16,,text_body_id,0.94,,10.17882/49388,,availability/archival,page_16,plain_text_with_block,2-4,Inline link (URL or DOI in text)
1,10.1002_ece3.4466,https://doi.org/10.5061/dryad.r6nq870,Primary,DATA ACCESSIBILITY The dataset supporting this...,,,,,5,DATA ACCESSIBILITY,da_hit_priority,0.98,Dryad,10.5061/dryad.r6nq870,,availability/archival,da_block,,2-5,Data availability section
2,10.1002_ece3.5260,https://doi.org/10.5061/dryad.2f62927,Primary,DATA AVAILABILITY DNA sequences: GenBank MK838...,,,,,13,DATA AVAILABILITY,da_hit_priority,0.98,Dryad,10.5061/dryad.2f62927,,availability/archival,da_block,,2-5,Data availability section
3,10.1002_ece3.6303,https://doi.org/10.5061/dryad.37pvmcvgb,Primary,DATA AVAILABILITY STATEMENT DNA sequences have...,,,,,11,DATA AVAILABILITY STATEMENT,da_hit_priority,0.98,Dryad,10.5061/dryad.37pvmcvgb,,availability/archival,da_block,,2-5,Data availability section
4,10.1002_ece3.9627,https://doi.org/10.5061/dryad.b8gtht7h3,Primary,DATA AVAILABILITY STATEMENT All R code and dat...,,,,,12,DATA AVAILABILITY STATEMENT,da_hit_priority,0.98,Dryad,10.5061/dryad.b8gtht7h3,,availability/archival,da_block,,2-5,Data availability section
5,10.1002_ecs2.1280,https://doi.org/10.5061/dryad.p3fg9,Primary,supportIng InforMAtIon Additional Supporting I...,,,,,16,supportIng InforMAtIon,da_hit_priority,0.98,Dryad,10.5061/dryad.p3fg9,,availability/archival,da_block,,2-5,Data availability section
6,10.1002_ecs2.4619,https://doi.org/10.25349/d9qw5x,Primary,DATA AVAILABILITY STATEMENT Data and novel cod...,,,,,10,DATA AVAILABILITY STATEMENT,da_hit_priority,0.98,,10.25349/d9qw5x,,availability/archival,da_block,,2-5,Data availability section
7,10.1002_esp.5058,https://doi.org/10.5061/dryad.jh9w0vt9t,Primary,DATA AVAILABILITY STATEMENT The dataset of the...,,,,,11,DATA AVAILABILITY STATEMENT,da_hit_priority,0.98,Dryad,10.5061/dryad.jh9w0vt9t,,availability/archival,da_block,,2-5,Data availability section
8,10.1002_esp.5090,https://doi.org/10.5066/p9353101,Secondary,The DEMs are available at https://doi.org/10.5...,The DEMs are available\nat https://doi.org/10....,,,,5,,text_body_id,0.94,,10.5066/p9353101,,availability/archival,page_5,plain_text_with_block,2-2,Numeric citation (Vancouver/IEEE)
9,10.1002_mp.14424,https://doi.org/10.7937/tcia.2020.6c7y-gq39,Primary,"In keeping with findable, accessible, interope...",,,,,,,refnum_superscript_guarded,0.99,,,,availability/archival,pymupdf_refnum,,2-2,Numeric citation (Vancouver/IEEE)


In [ ]:
print(out_df["in_text_span"][0])

In keeping with findable, accessible, interoperable, re-usable (FAIR) data usage principles,50 all PleThora thoracic cavity and pleural effusion segmentations have been made available on TCIA at https://doi.org/10.7937/tcia.2020.6c7ygq39.51 Thoracic cavity segmentations are in a compressed NIfTI format, are named for their respective case and reviewer (e.g., “LUNG1-001_thor_cav_primary_reviewer.nii.gz”), and are indexed in folders labeled after their respective NSCLC-Radiomics collection cases (e.g., “LUNG1-001”). Pleural effusion segmentations are likewise saved in a compressed NIfTI format and named for their respective case and reviewer (e.g., “LUNG1-001_effusion_first_reviewer.nii.gz”). Many thoracic cavity and all pleural effusion segmentations were reviewed by two or more experts. In this dataset’s original TCIA publication, only primary reviewer segmentations were made available. However, all reviewers’ segmentations — primary, secondary, and tertiary — were made available in a 

In [ ]:
# Colab cell — DOI -> exact ref# -> paragraphs that truly contain superscript/[n]/punctuation-guarded citations
# Span-aware detection + safe textual fallback (no false matches on decimals), continuation-merge, cleaning, locator line

import fitz, re
from statistics import median

# ------------------------- CONFIG -------------------------
PDF_PATH  = "/content/drive/MyDrive/Make_data_count_challenge/Data/train/PDF/10.1002_2017jc013030"
INPUT_DOI = "https://doi.org/10.17882/49388"   # change to DB1/DB2 as needed

def log(step, msg): print(f"[{step}] {msg}")

# ------------------------- DOI / REFS -------------------------
DOI_RX = re.compile(r"(10\.\d{4,9}/[-._;()/:A-Z0-9]+)", re.I)

def normalize_doi(s: str) -> str:
    s = s.strip().lower().replace("\u200b","").replace("\ufeff","").replace("\u00ad","")
    s = s.replace("https://doi.org/","").replace("http://doi.org/","")
    return re.sub(r"\s+","", s)

def find_references_page(doc):
    rx = re.compile(r"\bREFERENCES\b", re.I)
    for pno in range(len(doc)-1, -1, -1):
        if rx.search(doc[pno].get_text("text")): return pno
    raise RuntimeError("REFERENCES section not found")

def parse_references_strict(doc, start_page):
    # Only 1–3 digit reference numbers start a ref (prevents years like "2020.")
    all_lines = []
    for p in range(start_page, len(doc)):
        all_lines += doc[p].get_text("text").replace("\r","\n").split("\n")
    refs, cur_num, cur_buf = {}, None, []
    start_rx = re.compile(r"^\s*(\d{1,3})\.\s+(.*)$")
    def flush():
        nonlocal cur_num, cur_buf
        if cur_num and cur_buf:
            refs[cur_num] = " ".join(x.strip() for x in cur_buf if x.strip())
        cur_num, cur_buf = None, []
    for line in all_lines:
        m = start_rx.match(line)
        if m:
            flush(); cur_num = m.group(1); cur_buf = [m.group(2)]
        elif cur_num:
            cur_buf.append(line)
    flush()
    return refs

def doi_to_refnum_exact(ref_map: dict, doi_raw: str):
    target = normalize_doi(doi_raw)
    for num, entry in ref_map.items():
        for d in DOI_RX.findall(entry):
            if normalize_doi(d) == target:
                return num
    return None

# ------------------------- PAGE -> PARAGRAPHS (with lines & spans) -------------------------
def page_paragraphs_with_lines_and_spans(page, gap_factor=0.65):
    """
    Return a list of paragraphs; each paragraph is a dict:
      { "text": str, "lines": [ { "text": str, "spans": [ {text,size,bbox}, ... ] }, ... ] }
    """
    d = page.get_text("dict")
    paras = []
    for block in d.get("blocks", []):
        if block.get("type", 0) != 0: continue
        raw_lines = []
        for ln in block.get("lines", []):
            spans = ln.get("spans", [])
            if not spans: continue
            y0 = min(sp["bbox"][1] for sp in spans if "bbox" in sp)
            y1 = max(sp["bbox"][3] for sp in spans if "bbox" in sp)
            line_text = "".join(sp.get("text","") for sp in spans).replace("\r","")
            line_spans = [{"text": sp.get("text",""), "size": sp.get("size",0), "bbox": sp.get("bbox",[0,0,0,0])} for sp in spans]
            raw_lines.append((y0, y1, line_text, line_spans))
        if not raw_lines: continue
        heights = [y1-y0 for (y0,y1,_,_) in raw_lines]
        med_h = sorted(heights)[len(heights)//2] or 1.0

        cur_lines, cur_texts, prev_y1 = [], [], None
        for (y0,y1,ltxt,lspans) in raw_lines:
            if prev_y1 is None:
                cur_lines, cur_texts = [ {"text": ltxt, "spans": lspans} ], [ltxt]
            else:
                gap = y0 - prev_y1
                if gap > gap_factor * med_h:
                    paras.append({"text": "\n".join(cur_texts), "lines": cur_lines})
                    cur_lines, cur_texts = [ {"text": ltxt, "spans": lspans} ], [ltxt]
                else:
                    cur_lines.append({"text": ltxt, "spans": lspans})
                    cur_texts.append(ltxt)
            prev_y1 = y1
        if cur_lines:
            paras.append({"text": "\n".join(cur_texts), "lines": cur_lines})
    return [p for p in paras if p["text"].strip()]

# ------------------------- CONTINUATION MERGE -------------------------
def is_sentence_complete(s: str) -> bool:
    return re.search(r"[.!?][\"\')\]]*\s*$", s.strip()) is not None

def looks_like_continuation(line_text: str) -> bool:
    first = next((ln for ln in line_text.splitlines() if ln.strip()), "")
    if not first: return False
    if re.match(r"^[,.;:\-\)\]]", first): return True
    if re.match(r"^[a-z]", first): return True
    if len(first.split()) < 3: return True
    return False

def merge_continuations(paras):
    merged, i = [], 0
    while i < len(paras):
        cur = paras[i]
        j = i
        while (j+1 < len(paras)) and (not is_sentence_complete(cur["text"])) and looks_like_continuation(paras[j+1]["text"]):
            cur = {"text": cur["text"] + "\n" + paras[j+1]["text"],
                   "lines": cur["lines"] + paras[j+1]["lines"]}
            j += 1
        merged.append(cur)
        i = j + 1
    return merged

# ------------------------- SUPERSCRIPT / [n] / GUARDED-TEXT DETECTION -------------------------
def line_has_superscript_or_bracket_or_guarded_text(line, ref_num: str) -> bool:
    """
    True if the line contains one of:
      1) A separate smaller span whose text is exactly ref_num (superscript),
      2) A bracketed form like [ref_num],
      3) A punctuation-guarded textual citation (e.g., '.22,' or ')22;' or ']22,'),
         but **NOT** decimals (uses a negative digit lookbehind).
    """
    spans = line["spans"]
    line_text = line["text"]

    # 2) [ref_num]
    if re.search(rf"\[\s*{re.escape(ref_num)}\s*\]", line_text):
        return True

    # 1) superscript via smaller span
    if spans:
        sizes = [sp.get("size", 0) for sp in spans if sp.get("size",0) > 0]
        med_size = median(sizes) if sizes else 0
        small_thresh = med_size * 0.92 if med_size else 0
        for i, sp in enumerate(spans):
            if sp.get("text","").strip() == ref_num:
                size = sp.get("size", 0)
                if med_size and size and size < small_thresh:
                    prev_text = spans[i-1].get("text","") if i>0 else ""
                    prev_char = prev_text.rstrip()[-1:] if prev_text else ""
                    if prev_char in {".", ")", "]"} or prev_char == "":
                        return True

    # 3) punctuation-guarded textual fallback (handles ".22,23")
    #    Must NOT be preceded by a digit (avoids "1.22"), and must be followed by comma/semicolon/space.
    #    Accepts a leading ., ), ] with optional spaces.
    pat = re.compile(rf"(?<!\d)[\.\)\]]\s*{re.escape(ref_num)}(?=[,;])")
    if pat.search(line_text):
        return True

    return False

def paragraph_has_true_ref(paragraph_dict, ref_num: str):
    for idx, line in enumerate(paragraph_dict["lines"], start=1):
        if line_has_superscript_or_bracket_or_guarded_text(line, ref_num):
            return True, line["text"], idx
    return False, "", None

# ------------------------- CLEANING -------------------------
def clean_text_keep_lines(s: str) -> str:
    s = s.replace("\u200b","").replace("\ufeff","").replace("\u00ad","")
    s = re.sub(r"(\w)-\n(\w)", r"\1\2", s)               # de-hyphenate word-\nwraps
    s = "\n".join(re.sub(r"[ \t]+", " ", ln).rstrip() for ln in s.splitlines())
    return s.strip()

# ------------------------- PIPELINE -------------------------
doc = fitz.open(PDF_PATH)
log("OPEN", f"PDF loaded: {PDF_PATH}")

refs_start = find_references_page(doc)
log("REFS", f"References page index: {refs_start}  (1-based {refs_start+1})")

ref_map = parse_references_strict(doc, refs_start)
ref_num = doi_to_refnum_exact(ref_map, INPUT_DOI)
log("REFS", f"Resolved ref number for DOI '{INPUT_DOI}': {ref_num}")
if not ref_num:
    raise RuntimeError("Could not resolve reference number for DOI.")

body_pages_with_hits = 0
extracted = []  # (page_idx, para_idx_on_page, augmented_span)

for pno in range(refs_start):  # body only
    raw_paras = page_paragraphs_with_lines_and_spans(doc[pno])
    merged_paras = merge_continuations(raw_paras)

    any_hit_on_page = False
    for idx, para in enumerate(merged_paras, start=1):
        hit, loc_line, loc_idx = paragraph_has_true_ref(para, ref_num)
        if not hit:
            continue
        any_hit_on_page = True
        cleaned_par = clean_text_keep_lines(para["text"])
        cleaned_line = clean_text_keep_lines(loc_line) if loc_line else ""
        locator = (
            f"{INPUT_DOI} which is reference number {ref_num} indicated at this line {cleaned_line} of paragraph {idx}"
            if loc_idx is not None else
            f"{INPUT_DOI} which is reference number {ref_num} indicated in paragraph {idx}"
        )
        augmented = f"{cleaned_par}\n\n{locator}"
        extracted.append((pno, idx, augmented))
    if any_hit_on_page:
        body_pages_with_hits += 1

# ------------------------- OUTPUT -------------------------
print("\n=== Extraction Summary ===")
print(f"Input DOI                 : {INPUT_DOI}")
print(f"Resolved Reference Number : {ref_num}")
print(f"Body pages with hits      : {body_pages_with_hits}")
print(f"Paragraphs extracted      : {len(extracted)}")

for i, (pno, para_idx, span) in enumerate(extracted, 1):
    print(f"\n--- Paragraph {i} (Page {pno+1} • Paragraph #{para_idx}) ---\n")
    print(span)
print("\n--- End ---")

[OPEN] PDF loaded: /content/drive/MyDrive/Make_data_count_challenge/Data/train/PDF/10.1002_2017jc013030.pdf
[REFS] References page index: 16  (1-based 17)
[REFS] Resolved ref number for DOI 'https://doi.org/10.17882/49388': 5

=== Extraction Summary ===
Input DOI                 : https://doi.org/10.17882/49388
Resolved Reference Number : 5
Body pages with hits      : 0
Paragraphs extracted      : 0

--- End ---


In [ ]:
!pip install -q marker-pdf pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.1/188.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.0/221.0 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/

In [ ]:
# ===== 4-way THREADED Marker runner (spawn-free; safe in notebooks) =====
import os, re, glob, math, threading
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# --------- Paths (EDIT if needed) ---------
PDF_DIR   = "/content/drive/MyDrive/Make_data_count_challenge/Data/train/PDF"
CSV_PATH  = "/content/drive/MyDrive/Make_data_count_challenge/Data/train_labels_cleaned.csv"
OUT_MD    = "/content/drive/MyDrive/Make_data_count_challenge/marker_md"
OUT_CSV   = "/content/drive/MyDrive/Make_data_count_challenge/marker_doi_matches_gpu.csv"
os.makedirs(OUT_MD, exist_ok=True)

# --------- Torch hygiene (single process, many threads) ----------
import torch
torch.cuda.set_device(0)
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True
os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("OMP_NUM_THREADS","1")
os.environ.setdefault("MKL_NUM_THREADS","1")

# --------- Marker imports (ok at top in single process) ----------
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

# --------- DOI helpers (same accuracy as yours) ---------
DOI_CORE_RE   = re.compile(r"(10\.\d{4,9}/[^\s<>'\"(){}\[\]]+)", re.IGNORECASE)
BROKEN_DOI_RE = re.compile(r"(10\s*\.\s*\d{4,9}\s*/\s*[^<>'\"(){}\[\]\s]+(?:\s*[^<>'\"(){}\[\]])*)", re.IGNORECASE)
HEADING_RE    = re.compile(r"^(#{1,6})\s+(.+?)\s*$", re.MULTILINE)

def _squash_ws(s: str) -> str:
    return re.sub(r"\s+", "", s or "")

def normalize_doi(s: str) -> str:
    if not s: return ""
    t = s.strip().replace("\u200b", "")
    t = re.sub(r"(?i)\bdoi\s*[:=]\s*", "", t)
    t = re.sub(r"(?i)^https?://\s*doi\.org/\s*", "", t)
    m = DOI_CORE_RE.search(t) or BROKEN_DOI_RE.search(t)
    if m: t = m.group(1)
    t = _squash_ws(t).rstrip(").,;:").lower()
    return t

def extract_dois(text: str) -> set:
    found = set()
    for m in DOI_CORE_RE.finditer(text or ""):
        found.add(normalize_doi(m.group(0)))
    for m in BROKEN_DOI_RE.finditer(text or ""):
        found.add(normalize_doi(m.group(0)))
    return {d for d in found if d.startswith("10.")}

def split_markdown_into_sections(md: str):
    if not md or not HEADING_RE.search(md):
        return [{"level": 0, "title": "Full Document", "content": md or ""}]
    sections = []
    hits = [(m.start(), m.end(), len(m.group(1)), m.group(2)) for m in HEADING_RE.finditer(md)]
    for i, (s, e, lvl, title) in enumerate(hits):
        start = e
        end   = hits[i+1][0] if i+1 < len(hits) else len(md)
        sections.append({"level": lvl, "title": title.strip(), "content": md[start:end].strip("\n")})
    return sections

SECTION_BUCKETS = [
    ("References",        re.compile(r"\b(references|bibliography|works\s+cited)\b", re.I)),
    ("Data Availability", re.compile(r"\b(data\s+availability|availability\s+of\s+data|data\s+and\s+materials)\b", re.I)),
    ("Methods",           re.compile(r"\b(materials?\s+and\s+methods|methods?|methodology)\b", re.I)),
    ("Results",           re.compile(r"\bresults?\b", re.I)),
    ("Discussion",        re.compile(r"\bdiscussion\b", re.I)),
    ("Conclusions",       re.compile(r"\bconclusions?\b", re.I)),
    ("Supplementary",     re.compile(r"\b(supplementary|supplemental)\b", re.I)),
]
def bucket_section_name(raw_title: str) -> str:
    for name, rx in SECTION_BUCKETS:
        if rx.search(raw_title or ""):
            return name
    return raw_title or "(Unlabeled)"

def find_pdf_for_article(article_id: str) -> str:
    exact = os.path.join(PDF_DIR, f"{article_id}.pdf")
    if os.path.isfile(exact):
        return exact
    cands = glob.glob(os.path.join(PDF_DIR, f"*{re.escape(article_id)}*.pdf"))
    return cands[0] if cands else ""

# --------- Thread-local converter (one PdfConverter per thread) ---------
_tls = threading.local()
_models_global = None
_models_lock = threading.Lock()

def _get_thread_converter():
    global _models_global
    if getattr(_tls, "converter", None) is None:
        # Build shared model dict once (CPU side), then create a PdfConverter per thread
        with _models_lock:
            if _models_global is None:
                _models_global = create_model_dict()
        _tls.converter = PdfConverter(_models_global)
    return _tls.converter

# --------- Worker: process shard in this thread ----------
def _process_shard_thread(shard_idx: int, article_ids: list[str], df_all: pd.DataFrame):
    conv = _get_thread_converter()
    article_md_path  = {}
    article_sec_dois = {}

    # render each PDF -> md (reuse if exists), then parse
    for aid in article_ids:
        pdf = find_pdf_for_article(aid)
        if not pdf:
            article_md_path[aid]  = ""
            article_sec_dois[aid] = {}
            continue

        md_path = os.path.join(OUT_MD, f"{os.path.splitext(os.path.basename(pdf))[0]}.md")
        md = None
        if os.path.isfile(md_path):
            try:
                with open(md_path, "r", encoding="utf-8") as f:
                    md = f.read()
            except Exception:
                md = None

        if md is None:
            try:
                rendered = conv(pdf)         # heavy GPU call; concurrent across threads
                md, _, _ = text_from_rendered(rendered)
                md = md or ""
                with open(md_path, "w", encoding="utf-8") as f:
                    f.write(md)
            except Exception:
                article_md_path[aid]  = ""
                article_sec_dois[aid] = {}
                continue

        # parse sections/dois
        sec_map = {}
        for sec in split_markdown_into_sections(md):
            bucket = bucket_section_name(sec["title"])
            dois = extract_dois(sec["content"])
            if dois:
                sec_map.setdefault(bucket, set()).update(dois)

        article_md_path[aid]  = md_path
        article_sec_dois[aid] = sec_map

    # explode to triples
    triples = []
    for aid, sec_map in article_sec_dois.items():
        for bucket, dois in (sec_map or {}).items():
            for d in dois:
                triples.append((aid, bucket, d))
    secdf = pd.DataFrame(triples, columns=["article_id", "section_bucket", "doi_norm"])

    # restrict df to this shard's ids (for speed)
    df_shard = df_all[df_all["article_id"].isin(article_ids)].copy()

    if not secdf.empty and not df_shard.empty:
        secdf["article_id"] = secdf["article_id"].astype("category")
        secdf["doi_norm"]   = secdf["doi_norm"].astype("category")
        joined = df_shard.merge(
            secdf,
            left_on=["article_id", "dataset_doi_norm"],
            right_on=["article_id", "doi_norm"],
            how="left"
        )
        joined["found"] = joined["section_bucket"].notna()
        out = (joined
               .groupby(["article_id", "dataset_id", "dataset_doi_norm"], as_index=False)
               .agg(Section_reference_is_coming_from=("section_bucket",
                     lambda s: " | ".join(sorted(set([x for x in s.dropna()])))),
                    found=("found","any")))
    else:
        out = df_shard[["article_id", "dataset_id", "dataset_doi_norm"]].copy()
        out["found"] = False
        out["Section_reference_is_coming_from"] = ""

    # diagnostics/paths
    out["matched_doi"] = out.apply(lambda r: r["dataset_doi_norm"] if r["found"] else "", axis=1)
    out["n_sections_with_any_doi"] = out["article_id"].map(lambda aid: len(article_sec_dois.get(aid, {})))
    out["markdown_path"] = out["article_id"].map(lambda aid: article_md_path.get(aid, ""))

    shard_csv = OUT_CSV.replace(".csv", f".thread{shard_idx}.csv")
    out.to_csv(shard_csv, index=False)
    return shard_csv

# --------- Launcher: split into 4 shards, run 4 threads ----------
def run_four_way_threaded(gpu_threads: int = 4):
    df = pd.read_csv(CSV_PATH)
    if "article_id" not in df.columns or "dataset_id" not in df.columns:
        raise ValueError("CSV must contain columns: article_id, dataset_id")
    df["article_id"] = df["article_id"].astype(str)
    df["dataset_id"] = df["dataset_id"].astype(str)
    df["dataset_doi_norm"] = df["dataset_id"].map(normalize_doi)

    uniq = df["article_id"].dropna().unique().tolist()
    k = gpu_threads
    shards = [uniq[i::k] for i in range(k)]

    paths = []
    with ThreadPoolExecutor(max_workers=k) as ex:
        futs = {ex.submit(_process_shard_thread, i, shard, df): i for i, shard in enumerate(shards)}
        for fut in as_completed(futs):
            paths.append(fut.result())

    parts = [pd.read_csv(p) for p in sorted(paths)]
    out = pd.concat(parts, ignore_index=True)
    out = (out
           .sort_values(["article_id","dataset_id"])
           .drop_duplicates(subset=["article_id","dataset_id"], keep="first"))
    out.to_csv(OUT_CSV, index=False)
    print(f"Combined: {OUT_CSV}")

    try:
        from IPython.display import display
        display(out.head(20))
    except Exception:
        pass
    return out

# ==== RUN (no __main__ guard needed in notebooks) ====
marker_results_df = run_four_way_threaded(gpu_threads=2)  # set to 3 if you see OOM





























































Detecting bboxes: 100%|██████████| 1/1 [00:10<00:00, 10.06s/it]

Recognizing Text:   0%|          | 0/22 [00:00<?, ?it/s]
CRITICAL:concurrent.futures:Exception in initializer:
Traceback (most recent call last):
  File "/usr/lib/python3.12/concurrent/futures/process.py", line 243, in _process_worker
    initializer(*initargs)
  File "/usr/local/lib/python3.12/dist-packages/pdftext/extraction.py", line 39, in worker_init
    pdf_doc = _load_pdf(pdf_path, flatten_pdf)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pdftext/extraction.py", line 19, in _load_pdf
    pdf = pdfium.PdfDocument(pdf)
          ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pypdfium2/_helpers/document.py", line 78, in __init__
    self.raw, to_hold, to_close = _open_pdf(self._input, self._password, self._autoclose)
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^